# 08 — Fine-Tuned Backbone

Trains a ConvNeXt-T end to end instead of reading frozen DINOv2 features. Prior
RSNA competitions were won by fine-tuned ImageNet CNNs, so this is the arm most
likely to top the leaderboard, and the one the frozen path exists to be
measured against.

**Runs on Kaggle, not Colab.** Fine-tuning needs raw volumes, and one cache
shard is 9.7 GB. Pulling that to Colab failed twice at ~5.5 GB with
IncompleteRead -- a consistent ceiling rather than a flaky connection. On
Kaggle the cache is already mounted and nothing transfers. A T4 is slower than
an A100, but not slower than a download that never finishes.

**Attach:** one cache dataset, `knee-labels`, `knee_side_classifier`, and the
competition data. GPU on. The auxiliary report loss additionally wants
`report_embeddings.parquet` (built locally by `src/labels/report_embed.py`)
in the `knee-labels` dataset; without it that loss silently disables, which
the assemble cell prints.

**This run measures before it commits.** One study is ~5.5 series x 24 slices
= ~130 slice-forwards, so a full epoch over 4,349 studies is ~570,000. The
training cell reports actual throughput and projects the full sweep. Read that
before launching five folds.

Subsampling controls the cost, and both knobs are real trade-offs:

    SLICES_PER_SERIES  16 of 24     the reference geometry; 24 overfit for +0.002
    MAX_SERIES          5 of ~5.5   a finding may live only in a dropped series

The capacity ladder is finished (0.845 -> 0.869 -> 0.889 -> 0.891) and 16
slices x 5 series is where experiments iterate. Expect roughly **9 min/epoch,
~1.8 h for twelve epochs**. If it OOMs, drop `MAX_SERIES` to 4.

**Mirroring is used here**, unlike the frozen path: raw volumes can be
reflected, so left knees are mapped onto the canonical side along the axis the
acquisition plane dictates. That makes this a comparison of two things at once,
backbone and laterality handling, which is worth remembering when reading the
result.

## 1. Setup

In [ ]:
!pip install -q timm

import os
import os
import subprocess

# CUDA MUST NOT BE INITIALISED IN THIS PROCESS. Lightning's ddp_notebook
# strategy forks, and a fork inheriting a live CUDA context dies in the child
# with "Cannot re-initialize CUDA in forked subprocess".
#
# The trap is that the obvious check is the unsafe one. torch's own docs are
# explicit: is_available() "will NOT poison fork if
# PYTORCH_NVML_BASED_CUDA_CHECK=1" -- meaning by default it does, because it
# falls through to _cuda_getDeviceCount(), which calls cuInit. device_count()
# asks NVML first and stays clean. get_device_name and get_device_properties
# reach _lazy_init and are also out. Device names come from nvidia-smi, which
# is a separate process entirely.
#
# The env var is set as well, before torch is touched, so that any library
# calling is_available() downstream takes the NVML path too.
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"

import torch

DEVICES = torch.cuda.device_count()
if DEVICES == 0:
    raise SystemExit("Settings -> Accelerator -> GPU T4 x2")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("devices:", DEVICES, "| cuda initialised:", torch.cuda.is_initialized())
if DEVICES < 2:
    print("NOTE: one GPU visible. Settings -> Accelerator -> GPU T4 x2 "
          "roughly halves wall-clock. The run is correct either way -- "
          "ACCUM_STEPS below adapts to hold the effective batch constant.")

# Lightning ships in the Kaggle image, but the package name moved between
# major versions and a missing install should fail here rather than thirty
# minutes into the cell that matters.
try:
    import lightning.pytorch as pl
except ImportError:
    try:
        import pytorch_lightning as pl
    except ImportError:
        raise SystemExit("pip install lightning, then re-run this cell")
print("lightning:", pl.__version__)

## 2. Repo modules

In [ ]:
# Materialise the repo modules. Generated from source by
# tools/make_notebook.py -- do not edit here, edit src/ and regenerate.
import importlib, pathlib, sys

FILES = {
    'src/__init__.py': '',
    'src/config.py': '"""Central configuration: label names, cache geometry, and environment paths.\n\nEverything here is imported by both the training pipeline and the submission\nnotebook, so it must stay dependency-free apart from the standard library.\n"""\n\nimport os\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Labels\n# ---------------------------------------------------------------------------\n# Exact submission spelling. The apostrophe in "Baker\'s" and the spaces are\n# part of the column names -- do not normalise them anywhere that touches a\n# submission file.\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\n\nID_COL = "StudyInstanceUID"\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\n\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\n# any augmentation that mirrors the image must also swap these pairs.\nLATERAL_LABEL_PAIRS = [\n    ("Medial Meniscus", "Lateral Meniscus"),\n    ("Medial OA", "Lateral OA"),\n]\n\n# ---------------------------------------------------------------------------\n# Cache geometry\n# ---------------------------------------------------------------------------\n# Kept as constants so a higher-resolution variant is a config change rather\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\n# x 24,371 series ~= 30 GB, which fits Kaggle\'s 200 GB private dataset limit.\nCACHE_SIZE = 224          # in-plane resolution after resampling\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\nCACHE_DTYPE = "uint8"\n\n# Robust intensity normalisation percentiles. MRI has no standardised units,\n# so every volume is scaled against its own distribution.\nCLIP_PCT_LOW = 1.0\nCLIP_PCT_HIGH = 99.0\n\n# Canonical laterality: left knees are mirrored to match right knees so the\n# medial compartment always falls on the same side of the image.\nCANONICAL_SIDE = "R"\n\n# Mirroring is applied at load time, not baked into the cache.\n#\n# The side tag is blank on most series in this dataset and the series\n# descriptions are anonymised, so roughly 42% of series have no recoverable\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\n# would mean rebuilding all of it every time side detection improves -- and it\n# is expected to improve, via study-level propagation and then a classifier\n# trained on the series whose side is known. The cache therefore stores volumes\n# in their acquired orientation with the side recorded alongside, and the data\n# loader flips on read, which costs nothing.\nMIRROR_IN_CACHE = False\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\nCOMPETITION = "rsna-knee-abnormality-detection"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\n# What a competition data root must contain to be recognised.\n_MARKERS = ("train_series", "test_series")\n\n\ndef _detect_data_root() -> Path:\n    """Locate the competition data across Kaggle, Colab, and local runs.\n\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\'s input directory is\n    searched by content rather than by name: the mount is not reliably at\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\n    through to the local fallback, which then fails much later with a confusing\n    path in the message.\n    """\n    env = os.environ.get("KNEE_DATA_ROOT")\n    if env:\n        return Path(env)\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        exact = kaggle_input / COMPETITION\n        if exact.exists():\n            return exact\n        # Search two levels: Kaggle mounts competition data under\n        # /kaggle/input/competitions/<slug>/ as well as directly under\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\n        try:\n            for child in sorted(kaggle_input.iterdir()):\n                if not child.is_dir():\n                    continue\n                if any((child / m).exists() for m in _MARKERS):\n                    return child\n                for grandchild in sorted(child.iterdir()):\n                    if grandchild.is_dir() and any(\n                            (grandchild / m).exists() for m in _MARKERS):\n                        return grandchild\n        except OSError:\n            pass\n\n    return PROJECT_ROOT / "data"\n\n\nDATA_ROOT = _detect_data_root()\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\n\nTRAIN_CSV = META_DIR / "train.csv"\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\nTEST_CSV = META_DIR / "test.csv"\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\n\nINDEX_PARQUET = META_DIR / "index.parquet"\n\nON_KAGGLE = Path("/kaggle").exists()\n',
    'src/data/__init__.py': '',
    'src/data/dicom_io.py': '"""DICOM reading, geometric slice ordering, and laterality resolution.\n\nTwo operations here are load-bearing and fail silently if done naively:\n\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\n   slice normal derived from ImageOrientationPatient, which is defined by the\n   scanner geometry rather than by convention.\n\n2. **Laterality.** Four of the twelve labels are compartment-specific\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\n   on the opposite side of the frame from a right knee, so left knees must be\n   mirrored into a canonical orientation before a model can learn the\n   distinction.\n"""\n\nimport re\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport pydicom\n\n# Header fields pulled during the index scan. Kept explicit so the parquet\n# schema is stable and the scan stays header-only.\nHEADER_FIELDS = [\n    "StudyInstanceUID",\n    "SeriesInstanceUID",\n    "SOPInstanceUID",\n    "Rows",\n    "Columns",\n    "PixelSpacing",\n    "ImagePositionPatient",\n    "ImageOrientationPatient",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "ScanningSequence",\n    "SequenceVariant",\n    "SeriesDescription",\n    "ProtocolName",\n    "BodyPartExamined",\n    "Manufacturer",\n    "ManufacturerModelName",\n    "MagneticFieldStrength",\n    "ImageLaterality",\n    "Laterality",\n    "InstanceNumber",\n]\n\n# DICOM DS/IS values carry their original string representation, which would\n# land in the index as object dtype. Cast these explicitly.\nNUMERIC_FIELDS = {\n    "Rows",\n    "Columns",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "MagneticFieldStrength",\n    "InstanceNumber",\n}\n\n_LEFT_PATTERN = re.compile(\n    r"\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\b",\n    re.IGNORECASE,\n)\n_RIGHT_PATTERN = re.compile(\n    r"\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\b",\n    re.IGNORECASE,\n)\n\n\ndef read_header(path) -> dict:\n    """Read one DICOM\'s headers without decoding pixel data."""\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\n    row = {"path": str(path)}\n    for field in HEADER_FIELDS:\n        value = getattr(ds, field, None)\n        if value is None:\n            row[field] = None\n        elif isinstance(value, pydicom.multival.MultiValue):\n            # Geometry fields are numeric; ScanningSequence and friends are\n            # multi-valued strings, so fall back to a joined token.\n            try:\n                row[field] = [float(v) for v in value]\n            except (TypeError, ValueError):\n                row[field] = "\\\\".join(str(v) for v in value)\n        elif field in NUMERIC_FIELDS:\n            try:\n                row[field] = float(value)\n            except (TypeError, ValueError):\n                row[field] = None\n        elif isinstance(value, (int, float, str)):\n            row[field] = str(value) if isinstance(value, str) else value\n        else:\n            row[field] = str(value)\n    return row\n\n\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\n    """Unit normal of the image plane, from the two direction cosines.\n\n    Falls back to the z axis when orientation is missing, which keeps a\n    malformed series ordered by patient-superior position rather than crashing.\n    """\n    if orientation is None or len(orientation) != 6:\n        return np.array([0.0, 0.0, 1.0])\n\n    row_cosine = np.asarray(orientation[:3], dtype=float)\n    col_cosine = np.asarray(orientation[3:], dtype=float)\n    normal = np.cross(row_cosine, col_cosine)\n\n    magnitude = np.linalg.norm(normal)\n    if magnitude == 0:\n        return np.array([0.0, 0.0, 1.0])\n    return normal / magnitude\n\n\ndef slice_position(header: dict) -> float:\n    """Scalar position of a slice along its own stack axis."""\n    position = header.get("ImagePositionPatient")\n    if position is None or len(position) != 3:\n        # No geometry available; fall back to InstanceNumber so ordering is at\n        # least deterministic. The index scan flags these series separately.\n        number = header.get("InstanceNumber")\n        return float(number) if number is not None else 0.0\n\n    normal = slice_normal(header.get("ImageOrientationPatient"))\n    return float(np.dot(np.asarray(position, dtype=float), normal))\n\n\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\n    """Order a series\' slices along the acquisition axis."""\n    return sorted(headers, key=slice_position)\n\n\ndef has_geometry(headers: Sequence[dict]) -> bool:\n    """True when every slice carries the position and orientation tags."""\n    for header in headers:\n        position = header.get("ImagePositionPatient")\n        orientation = header.get("ImageOrientationPatient")\n        if position is None or len(position) != 3:\n            return False\n        if orientation is None or len(orientation) != 6:\n            return False\n    return True\n\n\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\n    """Determine whether a series images a left or right knee.\n\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\n    description fields, which carry side markers in several languages. Returns\n    "L", "R", or None when the side cannot be established -- callers must treat\n    None as unusable rather than guessing, since a wrong side silently swaps the\n    medial and lateral labels.\n    """\n    for header in headers:\n        for field in ("ImageLaterality", "Laterality"):\n            value = header.get(field)\n            if isinstance(value, str):\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\n                # often present-but-empty than absent. An empty string is not a\n                # side and must not be read as one.\n                token = value.strip().upper()\n                if token in ("L", "LEFT"):\n                    return "L"\n                if token in ("R", "RIGHT"):\n                    return "R"\n\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\n        for header in headers:\n            text = header.get(field)\n            if not isinstance(text, str):\n                continue\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\n            left = bool(_LEFT_PATTERN.search(spaced))\n            right = bool(_RIGHT_PATTERN.search(spaced))\n            # Ambiguous strings mentioning both sides tell us nothing.\n            if left and not right:\n                return "L"\n            if right and not left:\n                return "R"\n\n    return None\n\n\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\n\n    Applies the rescale slope and intercept when present. Slices whose in-plane\n    shape differs from the first are skipped rather than silently broadcast.\n    """\n    slices = []\n    reference_shape = None\n    skipped = 0\n\n    for path in paths:\n        try:\n            ds = pydicom.dcmread(str(path), force=True)\n            pixels = ds.pixel_array.astype(np.float32)\n        except Exception:  # noqa: BLE001\n            # A truncated or malformed file loses one slice, not the series.\n            # The dataset contains files whose declared pixel length exceeds\n            # what is actually stored, and dropping 23 good slices over one\n            # bad one costs a whole study its imaging.\n            skipped += 1\n            continue\n\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n        if slope != 1.0 or intercept != 0.0:\n            pixels = pixels * slope + intercept\n\n        if reference_shape is None:\n            reference_shape = pixels.shape\n        elif pixels.shape != reference_shape:\n            skipped += 1\n            continue\n\n        slices.append(pixels)\n\n    if not slices:\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\n\n    return np.stack(slices, axis=0)\n\n\ndef series_paths(series_dir) -> List[str]:\n    """All DICOM files under one series directory."""\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\n\n\ndef resolve_study_laterality(series_sides: dict) -> dict:\n    """Fill in unknown series sides from other series in the same study.\n\n    A knee MRI study images one knee, so a side established on any series\n    applies to all of them. This matters because the DICOM side tags are\n    populated inconsistently across the contributing institutions -- a study\n    can carry the tag on its coronal series and omit it on the sagittals.\n\n    Report text is deliberately not used as a source here. Reports exist only\n    for training studies, so recovering a side from them would mirror the\n    training volumes and leave the test volumes untouched -- a train/test skew\n    that is worse than the missing side it fixes.\n\n    Args:\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\n\n    Returns:\n        The same structure with unknown entries filled where the study allows.\n        Studies whose series disagree are left untouched, since a conflict\n        means the assumption of one knee per study does not hold there.\n    """\n    resolved = {}\n    for study_uid, series in series_sides.items():\n        known = {side for side in series.values() if side in ("L", "R")}\n\n        if len(known) == 1:\n            side = known.pop()\n            resolved[study_uid] = {uid: side for uid in series}\n        else:\n            # No evidence, or contradictory evidence. Guessing would silently\n            # swap the medial and lateral labels on four of twelve findings.\n            resolved[study_uid] = dict(series)\n\n    return resolved\n\n\ndef laterality_coverage(series_sides: dict) -> dict:\n    """Counts before and after study-level propagation, for auditing."""\n    before = sum(1 for s in series_sides.values() for v in s.values()\n                 if v in ("L", "R"))\n    total = sum(len(s) for s in series_sides.values())\n\n    filled = resolve_study_laterality(series_sides)\n    after = sum(1 for s in filled.values() for v in s.values()\n                if v in ("L", "R"))\n\n    conflicts = sum(\n        1 for s in series_sides.values()\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\n    )\n\n    return {\n        "total_series": total,\n        "known_before": before,\n        "known_after": after,\n        "recovered": after - before,\n        "conflicting_studies": conflicts,\n    }\n',
    'src/data/cache.py': '"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\n\nThe same code path runs during cache construction and at submission time. Any\ndivergence between the two produces a train/test preprocessing skew, which is\ninvisible in local validation and only shows up as a collapsed leaderboard\nscore -- so callers must import from here rather than reimplementing.\n\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\norientation it was acquired in. Mirroring to a canonical side happens at load\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport cv2\nimport numpy as np\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\n    """Scale a volume to uint8 using its own robust percentile range.\n\n    MRI carries no standardised units -- intensity depends on scanner, field\n    strength, coil, and whether fat suppression was applied -- so each volume is\n    scaled against its own distribution rather than a global constant.\n    """\n    finite = volume[np.isfinite(volume)]\n    if finite.size == 0:\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\n        # Flat or degenerate volume; nothing to stretch.\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\n    return (scaled * 255.0).round().astype(np.uint8)\n\n\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\n    """Resample every slice to size x size with area/linear interpolation."""\n    size = size or config.CACHE_SIZE\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\n\n    for i, plane in enumerate(volume):\n        # Area interpolation avoids the aliasing that bilinear introduces when\n        # downsampling 800px acquisitions; linear is correct when upsampling.\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\n\n    return output\n\n\ndef select_slices(count: int, target: int = None) -> List[int]:\n    """Indices of `target` slices spread evenly across a stack of `count`.\n\n    Short series are handled by repeating edge slices rather than zero-padding,\n    so the volume stays anatomically continuous.\n    """\n    target = target or config.CACHE_SLICES\n    if count <= 0:\n        raise ValueError("cannot select slices from an empty series")\n    if count == target:\n        return list(range(count))\n\n    positions = np.linspace(0, count - 1, target)\n    return [int(round(p)) for p in positions]\n\n\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\n\n    Mirroring a knee means reflecting through the patient\'s left-right axis,\n    and which array axis that corresponds to depends on the acquisition plane:\n\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\n\n    A blanket column flip is wrong for sagittal series. It leaves medial and\n    lateral exactly where they were -- they are separated across slices, not\n    across the image -- and instead reflects anterior-posterior, moving the\n    patella to the back of the knee.\n\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\n    """\n    left_right = np.array([1.0, 0.0, 0.0])\n\n    if orientation is None or len(orientation) != 6:\n        # No geometry to reason from. Columns is the common case (coronal and\n        # axial together outnumber sagittal), but the caller should prefer not\n        # to mirror at all when geometry is missing.\n        return 2\n\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\n    normal = dicom_io.slice_normal(orientation)\n\n    alignments = [\n        (abs(float(normal @ left_right)), 0),\n        (abs(float(height @ left_right)), 1),\n        (abs(float(width @ left_right)), 2),\n    ]\n    return max(alignments)[1]\n\n\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\n    """Mirror left knees so the medial compartment is always on one side.\n\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\n    Lateral OA). Without this the model would have to infer the side before it\n    could place a finding, from a cue that is not reliably in the image.\n\n    The axis reflected is chosen from the acquisition geometry -- see\n    mirror_axis -- because it differs between sagittal and coronal series.\n    """\n    if side is None:\n        # Unknown side: leave untouched and let the caller decide. Guessing\n        # would silently swap four of the twelve labels.\n        return volume\n    if side.upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = mirror_axis(orientation)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef build_series_volume(headers: Sequence[dict]) -> Dict:\n    """Full pipeline for one series: sort, decode, normalise, resize.\n\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\n    not by default. The recorded `laterality` is what the loader mirrors on.\n    Returns the volume plus the provenance needed to audit it later.\n    """\n    if not headers:\n        raise ValueError("no headers supplied")\n\n    ordered = dicom_io.sort_slices(headers)\n    first_header = ordered[0]\n    side = dicom_io.resolve_laterality(ordered)\n    geometry_ok = dicom_io.has_geometry(ordered)\n\n    indices = select_slices(len(ordered))\n    selected = [ordered[i] for i in indices]\n\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\n\n    # Slices skipped as unreadable leave the volume short of CACHE_SLICES, and\n    # the cache is a fixed-shape array. Re-select across whatever decoded so\n    # the depth is exact regardless of how many files were dropped.\n    if raw.shape[0] != config.CACHE_SLICES:\n        raw = raw[select_slices(raw.shape[0])]\n\n    resized = resize_volume(raw.astype(np.float32))\n    normalized = normalize_intensity(resized)\n\n    # Left as acquired by default; the loader mirrors using the recorded side.\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\n    volume = (canonicalize_side(normalized, side,\n                                first_header.get("ImageOrientationPatient"))\n              if config.MIRROR_IN_CACHE else normalized)\n\n    first = ordered[0]\n    return {\n        "volume": volume,\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\n        "n_slices_raw": len(ordered),\n        "laterality": side,\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\n                         and side.upper() != config.CANONICAL_SIDE),\n        "geometry_ok": geometry_ok,\n        "rows": int(first.get("Rows") or 0),\n        "cols": int(first.get("Columns") or 0),\n    }\n\n\ndef build_series_record(item):\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\n\n    Defined at module level deliberately. A worker defined inside a notebook\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\n    spawn start method and survives under fork only by accident. Keeping it\n    importable makes the cache build work the same way everywhere.\n\n    Failures are returned rather than raised: one unreadable series must not\n    end a ninety-minute run, and the manifest records what to investigate.\n    """\n    series_uid, headers = item\n    try:\n        result = build_series_volume(headers)\n        return series_uid, result["volume"], {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": result["StudyInstanceUID"],\n            "n_slices_raw": result["n_slices_raw"],\n            "laterality": result["laterality"],\n            "mirrored": result["mirrored"],\n            "geometry_ok": result["geometry_ok"],\n            "rows": result["rows"],\n            "cols": result["cols"],\n            "error": None,\n        }\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\n        return series_uid, None, {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": study_uid,\n            "n_slices_raw": len(headers) if headers else 0,\n            "laterality": None,\n            "mirrored": False,\n            "geometry_ok": False,\n            "rows": 0,\n            "cols": 0,\n            "error": str(exc)[:200],\n        }\n',
    'src/data/index.py': '"""Header-only scan of the DICOM tree into a single parquet index.\n\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\nthe cost is dominated by file-system traversal rather than decoding. The\nresulting index drives cache construction and lets series be filtered and\naudited without touching the 570 GB of imaging again.\n\nUsage (in a Kaggle notebook):\n\n    from src.data.index import build_index\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\n"""\n\nimport argparse\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterator, List, Optional\n\nimport pandas as pd\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef iter_dicom_paths(root) -> Iterator[Path]:\n    """Yield every .dcm under root/study/series/ without materialising a list."""\n    root = Path(root)\n    for study_dir in sorted(root.iterdir()):\n        if not study_dir.is_dir():\n            continue\n        for series_dir in sorted(study_dir.iterdir()):\n            if not series_dir.is_dir():\n                continue\n            for path in sorted(series_dir.glob("*.dcm")):\n                yield path\n\n\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\n    """Read headers for a batch of files in parallel.\n\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\n    released during file reads and a process pool would only add overhead.\n    """\n    def safe_read(path):\n        try:\n            return dicom_io.read_header(path)\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\n            return {"path": str(path), "error": str(exc)}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(safe_read, paths))\n\n    return pd.DataFrame(rows)\n\n\ndef build_index(\n    split: str = "train_series",\n    out_path: Optional[str] = None,\n    data_root=None,\n    workers: int = 8,\n    batch_size: int = 5000,\n    limit: Optional[int] = None,\n) -> pd.DataFrame:\n    """Scan one split\'s DICOM tree and write a parquet index.\n\n    Processes in batches so a long scan reports progress and so peak memory\n    stays flat regardless of how many files the split holds.\n    """\n    root = Path(data_root or config.DATA_ROOT) / split\n    if not root.exists():\n        raise FileNotFoundError(f"{root} does not exist")\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:  # progress is a convenience, never a requirement\n        tqdm = None\n\n    frames = []\n    batch: List[Path] = []\n    scanned = 0\n\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\n    # counts files without a total rather than forcing a full listing first,\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\n\n    for path in iter_dicom_paths(root):\n        batch.append(path)\n        if limit is not None and len(batch) + scanned >= limit:\n            break\n        if len(batch) >= batch_size:\n            frames.append(scan_paths(batch, workers=workers))\n            scanned += len(batch)\n            if progress is not None:\n                progress.update(len(batch))\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\n            else:\n                print(f"  scanned {scanned} files", flush=True)\n            batch = []\n\n    if batch:\n        frames.append(scan_paths(batch, workers=workers))\n        scanned += len(batch)\n        if progress is not None:\n            progress.update(len(batch))\n\n    if progress is not None:\n        progress.close()\n\n    if not frames:\n        raise RuntimeError(f"no DICOM files found under {root}")\n\n    index = pd.concat(frames, ignore_index=True)\n    print(f"scanned {scanned} files across {index[\'SeriesInstanceUID\'].nunique()} series")\n\n    if "error" in index.columns:\n        failures = int(index["error"].notna().sum())\n        if failures:\n            print(f"WARNING: {failures} files failed to parse")\n\n    if out_path:\n        os.makedirs(Path(out_path).parent, exist_ok=True)\n        index.to_parquet(out_path, index=False)\n        print(f"wrote {out_path}")\n\n    return index\n\n\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\n    """Per-series roll-up: slice count, geometry completeness, laterality.\n\n    Surfaces the series that need attention -- missing geometry, unresolvable\n    side -- before an hour of cache construction is spent on them.\n    """\n    records = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        records.append({\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "SeriesInstanceUID": series_uid,\n            "n_slices": len(headers),\n            "geometry_ok": dicom_io.has_geometry(headers),\n            "laterality": dicom_io.resolve_laterality(headers),\n            "rows": headers[0].get("Rows"),\n            "cols": headers[0].get("Columns"),\n        })\n\n    return pd.DataFrame(records)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--out", default=None)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--limit", type=int, default=None)\n    args = parser.parse_args()\n\n    build_index(split=args.split, out_path=args.out,\n                workers=args.workers, limit=args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/model/__init__.py': '',
    'src/model/side_classifier.py': '"""Predict whether a cached series images a left or right knee.\n\nThe DICOM side tag is blank on ~41% of series and the descriptions are\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\ncompartment-specific, and placing them requires knowing the side, so the\nremainder is predicted from the imaging itself.\n\nThis trains on the 14,403 series whose side is known and applies to the rest.\nBoth halves come from the same cache, so it behaves identically at submission\ntime -- unlike report text, which exists only for training studies.\n\n**Why the whole ordered stack rather than one representative slice.** For\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\nthe left-right axis IS the slice axis, so individual slices look much the same\neither way -- what differs is the direction of travel through the volume, which\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\nchannels, which preserves it.\n"""\n\nfrom typing import Optional, Sequence\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:  # the module is importable without torch for testing\n    torch = None\n    nn = object\n\nfrom src import config\n\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\n# so half resolution is ample and quadruples throughput.\nINPUT_SIZE = 112\n\nSIDES = ["L", "R"]\n\n\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\n\n    Slices become channels, keeping their order, so the network can use the\n    medial-to-lateral direction that distinguishes sides on sagittal series.\n    """\n    import cv2\n\n    if volumes.ndim == 3:\n        volumes = volumes[None]\n\n    count, depth, _, _ = volumes.shape\n    output = np.empty((count, depth, size, size), dtype=np.float32)\n\n    for i in range(count):\n        for j in range(depth):\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\n                                      interpolation=cv2.INTER_AREA)\n\n    return output / 255.0\n\n\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\n    """Mirror a prepared batch along each item\'s medial-lateral axis.\n\n    Used for augmentation: mirroring an input flips its label, which doubles\n    the training set and forces the model to key on anatomy rather than on any\n    incidental left/right imbalance in the source institutions.\n\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\n    in the prepared batch slices are dimension 1 and columns dimension 3.\n    """\n    output = np.empty_like(batch)\n    for i, axis in enumerate(plane_axis):\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\n    return output\n\n\nif torch is None:\n    # Importable without torch so the array handling above can be tested on a\n    # machine that never trains. The network itself needs the real thing.\n    SideNet = None\nelse:\n  class SideNet(nn.Module):\n      """A small convolutional classifier over slice-channel input.\n\n      Trained from scratch deliberately. It is a coarse two-class problem with\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\n      competition\'s stance on pretrained weights is still unconfirmed, which this\n      piece of the pipeline should not depend on.\n      """\n\n      def __init__(self, in_channels: int = None, width: int = 32):\n          super().__init__()\n          in_channels = in_channels or config.CACHE_SLICES\n\n          def block(inp, out, stride=2):\n              return nn.Sequential(\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n              )\n\n          self.features = nn.Sequential(\n              block(in_channels, width),        # 112 -> 56\n              block(width, width * 2),          # 56 -> 28\n              block(width * 2, width * 4),      # 28 -> 14\n              block(width * 4, width * 4),      # 14 -> 7\n          )\n          self.head = nn.Sequential(\n              nn.AdaptiveAvgPool2d(1),\n              nn.Flatten(),\n              nn.Dropout(0.2),\n              nn.Linear(width * 4, 1),\n          )\n\n      def forward(self, x):\n          return self.head(self.features(x)).squeeze(-1)\n\n\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\n    """Collapse per-series predictions to one side per study.\n\n    A study is one knee, so its series must agree. Voting across them is\n    strictly more reliable than any single series -- and the observed\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\n\n    Returns {study_uid: "L" | "R"}.\n    """\n    if len(predictions) != len(study_uids):\n        raise ValueError(\n            f"predictions and study_uids differ in length: "\n            f"{len(predictions)} vs {len(study_uids)}")\n    if confidences is None:\n        confidences = np.ones(len(predictions))\n    elif len(confidences) != len(predictions):\n        # zip would truncate silently, shifting weights onto the wrong series.\n        raise ValueError(\n            f"confidences length {len(confidences)} does not match "\n            f"{len(predictions)} predictions")\n\n    scores = {}\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\n\n    result = {}\n    for study, entries in scores.items():\n        total = sum(weight for _, weight in entries)\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\n        result[study] = "R" if mean >= 0.5 else "L"\n    return result\n\n\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\n    """Fraction correct, with predictions as probabilities of the right knee."""\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\n',
    'src/model/embed.py': '"""Frozen DINOv2 slice embeddings.\n\nRuns the backbone once over every cached slice and stores the result, so the\naggregation stack trains on cached features in minutes rather than hours. The\nsame code runs at submission time, which is the point -- a divergence between\nhow features are computed for training and for inference is invisible locally\nand fatal on the leaderboard.\n\n**Why frozen, and why first.** The competition scores a second prize track on\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\nthe aggregation design -- which is where the modelling actually happens -- be\niterated dozens of times a day instead of three.\n\n**What is stored per slice.** The CLS token concatenated with the patch tokens\nreduced by `POOL_GRID`:\n\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\n\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\n\nThe first model used grid 1, and its weakness appeared exactly where the mean\npredicts one: regional findings (effusion, OA, Baker\'s) averaged 0.865 on gold\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\ntear occupies two or three patches out of 256, so a mean over all of them\nburies it in 250-odd patches of normal tissue.\n\nWide embeddings must be projected before the aggregation stack -- see\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\nin it against 8.2M with a projection.\n"""\n\nfrom typing import Optional\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:\n    torch = None\n    nn = None\n\nfrom src import config\n\n# The registers variant. Registers exist to absorb the high-norm artefact\n# tokens that otherwise contaminate patch features -- precisely what gets\n# pooled here.\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\n\n# timm\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\n# position embeddings are interpolated rather than silently mismatched.\nINPUT_SIZE = config.CACHE_SIZE\n\n# DINOv2-S token width.\nTOKEN_DIM = 384\n\n# How the 16x16 patch grid is reduced before storage.\n#\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\n#\n# The mean is what the first model used, and its weakness showed up exactly\n# where predicted: regional findings averaged 0.865 on gold while focal ones\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\n# attention pooling to find it.\nPOOL_GRID = 1\n\n\ndef embed_dim(grid: int = None) -> int:\n    """Stored width per slice: CLS plus the pooled patch tokens."""\n    grid = POOL_GRID if grid is None else grid\n    tokens = 1 + (grid * grid if grid > 1 else 1)\n    return TOKEN_DIM * tokens\n\n\nEMBED_DIM = embed_dim()\n\n# ImageNet statistics, which DINOv2 was trained under.\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\n                  checkpoint_path: Optional[str] = None):\n    """A frozen DINOv2 feature extractor in eval mode.\n\n    `checkpoint_path` loads weights from disk, which the submission notebook\n    needs: internet is disabled there, so timm cannot fetch anything.\n    """\n    import timm\n\n    model = timm.create_model(\n        model_name,\n        pretrained=pretrained and checkpoint_path is None,\n        img_size=INPUT_SIZE,\n        num_classes=0,\n    )\n\n    if checkpoint_path:\n        state = torch.load(checkpoint_path, map_location="cpu")\n        state = state.get("state_dict", state)\n        model.load_state_dict(state, strict=False)\n\n    model.eval()\n    for parameter in model.parameters():\n        parameter.requires_grad = False\n    return model\n\n\ndef normalize(slices):\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\n\n    The patch sequence is reshaped back to its square grid before pooling, so\n    neighbouring patches are averaged with their neighbours rather than with\n    whatever happened to sit beside them in a flattened sequence.\n    """\n    import torch.nn.functional as F\n\n    count, n_patches, channels = patches.shape\n    side = int(round(n_patches ** 0.5))\n    if side * side != n_patches:\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\n        return patches.mean(dim=1)\n\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\n    pooled = F.adaptive_avg_pool2d(square, grid)\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\n\n\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\n\n    Uses forward_features so the patch tokens are reachable at all; the model\'s\n    default pooled output discards them.\n    """\n    grid = POOL_GRID if grid is None else grid\n\n    with torch.no_grad():\n        tokens = encoder.forward_features(normalize(slices))\n\n        if tokens.ndim == 2:\n            # A backbone that already pools; repeat so the width is predictable.\n            repeats = embed_dim(grid) // tokens.shape[-1]\n            return tokens.repeat(1, repeats)\n\n        cls = tokens[:, 0]\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\n        patches = tokens[:, prefix:]\n\n        if grid > 1:\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\n\n\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\n\n    KneeNet expects a module mapping slices to features. When features are\n    already computed, the "backbone" is an identity that reshapes -- which\n    keeps one model definition for both the cached and the end-to-end path.\n    """\n\n    def __init__(self, feature_dim: int = EMBED_DIM):\n        super().__init__()\n        self.feature_dim = feature_dim\n\n    def forward(self, x):\n        return x\n',
    'src/model/dataset.py': '"""Study-level dataset over the cached volumes.\n\nJoins four sources into one training sample per study:\n\n    cache shards      the uint8 volumes, memory-mapped\n    manifest          which shard row holds which series\n    fused labels      12 soft targets and 12 sample weights per study\n    side predictions  L/R per series, from the tag or the classifier\n\nA sample is a whole study, not a series. The twelve findings are study-level\nlabels, and a single series often cannot support them -- effusion reads off a\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\nhas to see the study\'s series together.\n\n**Laterality is handled two ways, depending on the path.**\n\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\nside at load time, along the axis the acquisition plane dictates: slice order\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\nand axial, where it lies in-plane. Doing it at load time rather than in the\ncache means side detection can improve without rebuilding 29 GB.\n\nFor the frozen path this is not available -- the embeddings are computed from\nthe volume, so the image can no longer be flipped. There the side travels to\nthe model as a covariate (`is_right`, `side_known`) instead. That still\naddresses the original concern, which was that the model should not have to\n*infer* the side from cues that are not reliably present; it is handed the\nanswer either way. Which of the two performs better is an experiment, not a\nsettled question.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\nPLANE_MIRROR_AXIS = {\n    "Sagittal": 0,   # medial-lateral is the slice axis\n    "Coronal": 2,    # medial-lateral lies in-plane\n    "Axial": 2,\n}\nDEFAULT_MIRROR_AXIS = 2\n\n# Series metadata fed to the model as conditioning. Findings are\n# sequence-specific, and these three are given in both train and test.\n# `is_right` and `side_known` carry laterality as a covariate rather than\n# normalising it away by mirroring. With precomputed frozen embeddings the\n# image can no longer be flipped -- the features are already computed -- so the\n# side is handed to the model instead, which addresses the original concern\n# (the model should not have to *infer* the side) by a different route.\n# Mirroring remains available for the end-to-end path and is an A/B to run,\n# not a settled question.\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\n                "is_coronal", "is_axial", "is_right", "side_known"]\n\n\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\n                  plane: Optional[str]) -> np.ndarray:\n    """Reflect a left knee onto the canonical side, along the correct axis."""\n    if side is None or (isinstance(side, float) and np.isnan(side)):\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\n        # on four of the twelve findings.\n        return volume\n    if str(side).upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef series_metadata(row: pd.Series) -> np.ndarray:\n    """Conditioning vector for one series.\n\n    Fluid sensitivity and fat suppression are kept separate: the data\n    description notes they are often correlated but not equivalent, so\n    collapsing them would discard the cases where they disagree.\n    """\n    plane = str(row.get("Anatomical_Plane", ""))\n    side = row.get("side_predicted")\n    known = isinstance(side, str) and side.upper() in ("L", "R")\n    return np.array([\n        float(row.get("Fluid_Sensitive", 0) or 0),\n        float(row.get("Fat_Suppression", 0) or 0),\n        float(plane == "Sagittal"),\n        float(plane == "Coronal"),\n        float(plane == "Axial"),\n        float(known and side.upper() == "R"),\n        float(known),\n    ], dtype=np.float32)\n\n\nclass StudyIndex:\n    """Everything needed to assemble a study, resolved once up front.\n\n    Built from frames rather than files so it can be constructed in a test\n    without touching disk.\n    """\n\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\n                 labels: Optional[pd.DataFrame] = None,\n                 sides: Optional[pd.DataFrame] = None,\n                 max_series: Optional[int] = None,\n                 reports: Optional[pd.DataFrame] = None):\n        frame = manifest.merge(\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\n                         "Fat_Suppression", "Anatomical_Plane"]],\n            on="SeriesInstanceUID", how="left")\n\n        if sides is not None:\n            frame = frame.merge(\n                sides[["SeriesInstanceUID", "side_predicted"]],\n                on="SeriesInstanceUID", how="left")\n        else:\n            frame["side_predicted"] = frame.get("laterality_resolved")\n\n        # Drop series whose volume failed to build; their row in the cache is\n        # zeros and would train the model on blank images.\n        if "error" in frame.columns:\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\n\n        self.series = frame\n        self.max_series = max_series\n        self.labels = labels\n\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\n        self._by_study = {\n            study: group.reset_index(drop=True)\n            for study, group in frame.groupby(config.ID_COL)\n        }\n\n        if labels is not None:\n            keyed = labels.set_index(config.ID_COL)\n            self.studies = [s for s in self.studies if s in keyed.index]\n            self._labels = keyed\n        else:\n            self._labels = None\n\n        # Report sentence embeddings, a training-time-only signal: reports\n        # exist for train studies and never for test, so these feed an\n        # auxiliary loss rather than the model input. Stored as one float32\n        # matrix rather than a DataFrame so __getitem__ is an array row.\n        if reports is not None:\n            keyed_reports = reports.set_index(config.ID_COL)\n            self._report_rows = {study: i for i, study\n                                 in enumerate(keyed_reports.index)}\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\n            self.report_dim = self._report_matrix.shape[1]\n        else:\n            self._report_rows = None\n            self._report_matrix = None\n            self.report_dim = None\n\n    def __len__(self) -> int:\n        return len(self.studies)\n\n    def series_for(self, study: str) -> pd.DataFrame:\n        """Series of a study, capped deterministically when a cap is set.\n\n        The cap exists for the efficiency track: inference cost scales with\n        series count, and a study with a dozen series contributes far more\n        runtime than one with three. Selection is by position rather than at\n        random so a prediction is reproducible.\n        """\n        rows = self._by_study[study]\n        if self.max_series is not None and len(rows) > self.max_series:\n            keep = np.linspace(0, len(rows) - 1, self.max_series).round().astype(int)\n            rows = rows.iloc[np.unique(keep)].reset_index(drop=True)\n        return rows\n\n    def targets_for(self, study: str):\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\n        if self._labels is None or study not in self._labels.index:\n            return None, None\n        row = self._labels.loc[study]\n        targets = np.array([float(row[name]) for name in config.LABELS],\n                           dtype=np.float32)\n        weights = np.array(\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\n            dtype=np.float32)\n        return targets, weights\n\n    def report_for(self, study: str) -> Optional[np.ndarray]:\n        """The study\'s report embedding, or None when unavailable."""\n        if self._report_rows is None or study not in self._report_rows:\n            return None\n        return self._report_matrix[self._report_rows[study]]\n\n\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study: mirrored volumes, conditioning, targets, weights."""\n    rows = index.series_for(study)\n\n    volumes, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        volume = np.asarray(shard[int(row["position"])])\n        volume = mirror_volume(volume, row.get("side_predicted"),\n                               row.get("Anatomical_Plane"))\n        volumes.append(volume)\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(volumes) if volumes else np.zeros(\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\n            dtype=np.uint8),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef load_study_features(index: StudyIndex, study: str,\n                        shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study from precomputed slice embeddings.\n\n    Mirroring is unavailable here -- the features were computed from the image\n    and cannot be flipped after the fact -- so laterality reaches the model\n    through `series_metadata` instead.\n    """\n    rows = index.series_for(study)\n\n    features, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\n                                                                dtype=np.float32),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\n    """Pad a batch of studies to the largest series count, with a mask.\n\n    Series counts vary from one to a dozen, so the batch is padded and masked\n    rather than truncated. The mask must reach the attention pooling: without\n    it, padding rows are averaged in as if they were real series.\n    """\n    import torch\n\n    longest = max(len(item["volumes"]) for item in batch)\n\n    # Per-series shape is taken from the data rather than assumed, so the same\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\n                     None)\n    if reference is None:\n        raise ValueError("every study in the batch is empty")\n    per_series = reference.shape[1:]\n    dtype = reference.dtype\n\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\n    mask = np.zeros((len(batch), longest), dtype=bool)\n\n    for i, item in enumerate(batch):\n        count = len(item["volumes"])\n        if count:\n            volumes[i, :count] = item["volumes"]\n            meta[i, :count] = item["meta"]\n            mask[i, :count] = True\n\n    out = {\n        config.ID_COL: [item[config.ID_COL] for item in batch],\n        "volumes": torch.from_numpy(volumes),\n        "meta": torch.from_numpy(meta),\n        "mask": torch.from_numpy(mask),\n    }\n\n    if batch[0]["targets"] is not None:\n        out["targets"] = torch.from_numpy(\n            np.stack([item["targets"] for item in batch]))\n        out["weights"] = torch.from_numpy(\n            np.stack([item["weights"] for item in batch]))\n\n    # Report embeddings, present only where the index carries them. Studies\n    # without one get a zero row and a False mask so the auxiliary loss can\n    # exclude them without a ragged batch.\n    reports = [item.get("report") for item in batch]\n    dims = [r.shape[0] for r in reports if r is not None]\n    if dims:\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\n        report_mask = np.zeros(len(batch), dtype=bool)\n        for i, row in enumerate(reports):\n            if row is not None:\n                report[i] = row\n                report_mask[i] = True\n        out["report"] = torch.from_numpy(report)\n        out["report_mask"] = torch.from_numpy(report_mask)\n    return out\n',
    'src/model/net.py': '"""Study-level model: slices to series to study to twelve logits.\n\nThe backbone is deliberately the least interesting part. It is swappable, and\nwhich one wins is settled by measurement rather than by argument -- at 4,349\nstudies with mined labels, the aggregation above it matters more.\n\n    (B, S, D, H, W) uint8 volumes\n      -> backbone over slices          (B*S*D, F)\n      -> attention pool over slices    (B*S, F)\n      -> FiLM on sequence metadata     (B*S, F)\n      -> transformer over series       (B, F)      masked\n      -> head                          (B, 12)\n\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\nand Baker\'s cysts read off fluid-sensitive series, cartilage loss off\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\nevery series through one undifferentiated path throws away free signal.\n\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\nrows must be excluded from both the attention pool and the transformer, or a\nshort study is diluted by rows of zeros that the model reads as real series.\n"""\n\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom src import config\nfrom src.model.dataset import META_COLUMNS\n\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\n# backbone expects. Gives each slice a little through-plane context for free.\nSLICE_CONTEXT = 3\n\n\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\n\n    Edge slices repeat, which is preferable to zero padding: a black plane at\n    the edge of a volume is a feature the model would learn to key on.\n    """\n    n, depth, height, width = volumes.shape\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\n    stacked = torch.stack([previous, volumes, following], dim=2)\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\n\n\nclass AttentionPool(nn.Module):\n    """Masked attention pooling over a sequence of embeddings.\n\n    Preferred to mean pooling because the informative slices are a minority --\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\n    dilutes it by the rest.\n    """\n\n    def __init__(self, dim: int, hidden: int = 128):\n        super().__init__()\n        self.score = nn.Sequential(\n            nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n\n    def forward(self, x: torch.Tensor,\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\n        # x: (B, N, F), mask: (B, N) with True for real entries.\n        weights = self.score(x).squeeze(-1)\n        if mask is not None:\n            weights = weights.masked_fill(~mask, float("-inf"))\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\n        return (x * weights).sum(dim=1)\n\n\nclass FiLM(nn.Module):\n    """Feature-wise modulation of series embeddings by sequence metadata."""\n\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\n        super().__init__()\n        self.to_scale_shift = nn.Sequential(\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\n            nn.Linear(dim, dim * 2))\n\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\n        return x * (1.0 + scale) + shift\n\n\nclass KneeNet(nn.Module):\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\n\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\n    a frozen encoder is supported and is the intended first configuration: with\n    the trunk frozen, slice embeddings are computed once and the stack above\n    trains in minutes rather than hours.\n    """\n\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\n                 n_labels: int = None, series_layers: int = 2,\n                 n_heads: int = 8, dropout: float = 0.2,\n                 freeze_backbone: bool = False,\n                 input_dim: Optional[int] = None):\n        super().__init__()\n        # None is valid on the frozen path, where the trunk ran offline.\n        self.backbone = backbone if backbone is not None else nn.Identity()\n        self.feature_dim = feature_dim\n        n_labels = n_labels or len(config.LABELS)\n\n        if freeze_backbone:\n            for parameter in self.backbone.parameters():\n                parameter.requires_grad = False\n\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\n        # running the whole stack at that width would put ~85M parameters in\n        # FiLM alone. A projection keeps the aggregation at a sane size.\n        self.project = (nn.Linear(input_dim, feature_dim)\n                        if input_dim and input_dim != feature_dim\n                        else nn.Identity())\n\n        self.slice_pool = AttentionPool(feature_dim)\n        self.film = FiLM(feature_dim)\n\n        layer = nn.TransformerEncoderLayer(\n            d_model=feature_dim, nhead=n_heads,\n            dim_feedforward=feature_dim * 2, dropout=dropout,\n            batch_first=True, norm_first=True)\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\n        self.study_pool = AttentionPool(feature_dim)\n\n        self.head = nn.Sequential(\n            nn.LayerNorm(feature_dim),\n            nn.Dropout(dropout),\n            nn.Linear(feature_dim, n_labels))\n\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\n        count, depth = volumes.shape[0], volumes.shape[1]\n        if volumes.dtype == torch.uint8:\n            volumes = volumes.float().div_(255.0)\n\n        slices = stack_2p5d(volumes)\n        features = self.backbone(slices)\n        features = features.reshape(count, depth, self.feature_dim)\n        return self.slice_pool(features)\n\n    def pool_study(self, series: torch.Tensor, meta: torch.Tensor,\n                   mask: torch.Tensor) -> torch.Tensor:\n        """(B, S, F) series embeddings -> (B, F) one embedding per study.\n\n        The study representation the head reads. Exposed separately so\n        auxiliary objectives (report-embedding regression) can supervise the\n        representation without their parameters entering this module -- the\n        exported state dict must stay loadable by a bare `KneeNet`.\n        """\n        series = self.film(series, meta)\n\n        # The transformer takes True for positions to IGNORE, the inverse of\n        # our mask.\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\n        # A fully padded row emits NaN from the attention; zero it before\n        # pooling so one short study cannot poison the batch.\n        encoded = torch.nan_to_num(encoded)\n\n        return self.study_pool(encoded, mask=mask)\n\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\n                  mask: torch.Tensor) -> torch.Tensor:\n        """(B, S, F) series embeddings -> (B, n_labels).\n\n        Everything above the backbone. Shared by the end-to-end path and the\n        frozen path so the two cannot drift into different architectures.\n        """\n        return self.head(self.pool_study(series, meta, mask))\n\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\n                         mask: torch.Tensor) -> torch.Tensor:\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\n\n        The backbone has already run, so this starts at the slice pooling.\n        """\n        batch, series, depth, _ = features.shape\n        projected = self.project(features.reshape(batch * series, depth, -1))\n        pooled = self.slice_pool(projected)\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\n                              meta, mask)\n\n    def forward_pooled(self, volumes: torch.Tensor, meta: torch.Tensor,\n                       mask: torch.Tensor):\n        """End-to-end path returning (logits, pooled study embedding).\n\n        The training loop wants both: logits for the label losses, the pooled\n        representation for auxiliary supervision. Computing them together\n        avoids a second pass through the backbone.\n        """\n        batch, series = volumes.shape[0], volumes.shape[1]\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\n        pooled = self.pool_study(embeddings, meta, mask)\n        return self.head(pooled), pooled\n\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\n                mask: torch.Tensor) -> torch.Tensor:\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\n        logits, _ = self.forward_pooled(volumes, meta, mask)\n        return logits\n\n\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\n                 weights: torch.Tensor) -> torch.Tensor:\n    """BCE against soft targets, weighted per study and finding.\n\n    Targets are probabilities rather than 0/1 -- they come from two extractors\n    that disagree often enough that thresholding would teach the model to\n    reproduce extraction errors. Weights carry how much each target is\n    believed: full for agreement, less for conflict, least for silence.\n    """\n    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")\n    total = weights.sum()\n    if total <= 0:\n        return loss.mean() * 0.0\n    return (loss * weights).sum() / total\n\n\ndef pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\n                       ref_scores: torch.Tensor, ref_targets: torch.Tensor,\n                       weights: Optional[torch.Tensor] = None,\n                       ref_weights: Optional[torch.Tensor] = None,\n                       min_gap: float = 0.25) -> torch.Tensor:\n    """Per-finding RankNet loss between a batch and a reference set.\n\n    The metric is macro AUC -- a ranking statistic -- while BCE optimises\n    calibration; this term optimises the ordering directly. Pairs are formed\n    per finding between the current batch (`scores`/`targets`, (B, L), with\n    gradient) and a reference set ((Q, L), typically detached scores from\n    recent steps), because at one study per rank there are no pairs *within*\n    a batch to rank.\n\n    Only pairs whose soft targets differ by at least `min_gap` count: the\n    targets are extractor probabilities, and ordering two studies the\n    extractor itself barely distinguishes would train the model to reproduce\n    extraction noise. Surviving pairs are weighted by the target gap and by\n    both studies\' label weights, so a confident 0.9-vs-0.1 pair outweighs a\n    marginal one. Returns a zero that still touches `scores` when no pair\n    qualifies, so the graph is identical either way.\n    """\n    zero = scores.sum() * 0.0\n    if ref_scores.numel() == 0:\n        return zero\n\n    gap = targets.unsqueeze(1) - ref_targets.unsqueeze(0)      # (B, Q, L)\n    pair_weight = gap.abs() * (gap.abs() >= min_gap)\n    if weights is not None and ref_weights is not None:\n        pair_weight = (pair_weight * weights.unsqueeze(1)\n                       * ref_weights.unsqueeze(0))\n\n    total = pair_weight.sum()\n    if total <= 0:\n        return zero\n\n    margin = scores.unsqueeze(1) - ref_scores.unsqueeze(0)     # (B, Q, L)\n    loss = F.softplus(-torch.sign(gap) * margin)\n    return (loss * pair_weight).sum() / total\n\n\ndef load_knee_state(path, map_location="cpu") -> dict:\n    """Read a `KneeNet` state dict from either checkpoint format.\n\n    The end-to-end path writes two things over its life: a bare state dict\n    from `torch.save(model.state_dict(), ...)`, and a Lightning checkpoint,\n    which nests the same tensors under `state_dict` and prefixes every key\n    with `model.` because `KneeNet` is an attribute of the wrapper. The\n    submission must not care which it was handed -- requiring a manual\n    conversion step is requiring a step that can be forgotten, and it was.\n\n    Lightning checkpoints also carry optimiser state and callback objects,\n    which `weights_only=True` refuses, so that is attempted first and relaxed\n    only on failure. These are files this project wrote.\n    """\n    try:\n        obj = torch.load(path, map_location=map_location, weights_only=True)\n    except TypeError:       # torch too old to know the argument\n        obj = torch.load(path, map_location=map_location)\n    except Exception:       # noqa: BLE001 - a Lightning checkpoint\n        obj = torch.load(path, map_location=map_location, weights_only=False)\n\n    state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj\n    if any(key.startswith("model.") for key in state):\n        state = {key[len("model."):]: value for key, value in state.items()\n                 if key.startswith("model.")}\n    if not state:\n        raise ValueError(f"no weights found in {path}")\n    return state\n',
    'src/model/splits.py': '"""Cross-validation splits and the gold holdout.\n\nTwo rules govern every split here, and both exist because the obvious\nalternative produces a number that looks better and means less.\n\n**Split by study, never by series.** A study\'s ~5.5 series image the same knee\nin the same session. A random series split puts near-duplicates on both sides\nand reports an accuracy the model does not have.\n\n**The 58 gold studies never enter training.** They are the only radiologist\nlabels in the dataset and the only honest measurement available. Everything\nelse is trained on mined labels, and a model evaluated against mined labels can\nscore well by reproducing the extractor\'s mistakes. `assert_no_gold_leakage`\nis called by the training entrypoint rather than trusted to discipline.\n"""\n\nfrom typing import Iterator, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\nN_FOLDS = 5\nSEED = 0\n\n\ndef load_gold_uids(train_csv=None) -> set:\n    """Study UIDs of the 58 radiologist-labelled studies."""\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\n    labelled = frame[frame[config.LABELS[0]].notna()]\n    return set(labelled[config.ID_COL])\n\n\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\n    """Fail loudly if a gold study appears in training data.\n\n    Called from the training entrypoint. The failure this guards against is\n    silent -- a leaked gold study inflates the one number being used to decide\n    whether the model works.\n    """\n    leaked = set(train_uids) & gold_uids\n    if leaked:\n        raise ValueError(\n            f"{len(leaked)} gold studies leaked into training: "\n            f"{sorted(leaked)[:3]}..."\n        )\n\n\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\n    """A coarse key for balancing folds across a 12-label problem.\n\n    Full multi-label stratification over 2^12 combinations would leave most\n    cells empty at 4,349 studies. The count of positive findings captures the\n    axis that matters -- studies with many findings are rare and should not\n    cluster in one fold.\n    """\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\n    return positives.clip(upper=4)\n\n\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\n               seed: int = SEED) -> pd.DataFrame:\n    """Assign each study to a fold, stratified by positive-finding count.\n\n    Returns the frame with a `fold` column added.\n    """\n    frame = labels.copy().reset_index(drop=True)\n    key = stratification_key(frame)\n    rng = np.random.default_rng(seed)\n\n    frame["fold"] = -1\n    for value in key.unique():\n        rows = np.flatnonzero(key.to_numpy() == value)\n        rng.shuffle(rows)\n        frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\n\n    if (frame["fold"] < 0).any():\n        raise RuntimeError("some studies were not assigned a fold")\n    return frame\n\n\ndef iter_folds(labels: pd.DataFrame,\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\n    """Yield (train, validation) frames for each fold."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    for fold in range(n_folds):\n        validation = labels[labels["fold"] == fold]\n        train = labels[labels["fold"] != fold]\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\n\n\ndef series_for_studies(manifest: pd.DataFrame,\n                       study_uids: Sequence[str]) -> pd.DataFrame:\n    """All cached series belonging to a set of studies."""\n    wanted = set(study_uids)\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\n\n\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    rows = []\n    for fold in range(n_folds):\n        subset = labels[labels["fold"] == fold]\n        row = {"fold": fold, "studies": len(subset)}\n        for name in config.LABELS:\n            row[name] = float((subset[name] > 0.5).mean())\n        rows.append(row)\n    return pd.DataFrame(rows).set_index("fold")\n',
    'src/labels/__init__.py': '',
    'src/model/lightning.py': '"""Lightning wrapper for the end-to-end fine-tuning run.\n\nKaggle offers a T4 x2 accelerator, and the training loop is GPU-bound at\n~220 images/second, so a second device is close to a free halving of\nwall-clock. Lightning is here for the distributed plumbing, not for the\nabstraction: the semantics below are a faithful port of the hand-written loop\nin notebook 08, and any difference in the resulting score is a bug rather\nthan a design choice.\n\n**Everything in this module lives here rather than in a notebook cell.**\nLightning\'s `ddp_notebook` strategy forks, so children inherit memory and the\nmodule is not pickled on that path -- but dataloader workers and the spawn\nstart method both do pickle, and a class defined in a notebook cell belongs to\n`__main__` and does not survive that. Keeping these definitions importable as\n`src.model.lightning` makes the module correct under either start method; the\nbootstrap writes the files to disk before importing them. The same rule put\nthe cache\'s process-pool worker in `src/data/cache.py`.\n\nA related consequence of fork, and the reason the notebook is careful about\nit: **CUDA must not be initialised before `Trainer.fit`**. A fork inheriting a\nlive CUDA context fails in the child. `torch.cuda.get_device_name` and\n`get_device_properties` both call `_lazy_init`; `device_count` and\n`is_available` do not.\n\nThree DDP hazards are handled explicitly, because each one runs clean and\nsilently scores worse:\n\n**Memory-mapped shards are never serialised.** The cache is ~29 GB across\nthree memmaps. Holding them as attributes would carry the array bytes into any\nprocess the dataset is pickled into. They are opened lazily per process and\ndropped from `__getstate__`, which also means each dataloader worker maps the\nfiles itself rather than sharing a handle across a fork.\n\n**AUC is computed over gathered predictions.** It is a ranking metric, so\naveraging two per-rank AUCs is not the AUC of the whole set. Predictions and\ntargets are all-gathered before scoring. The distributed sampler pads the\nfinal batch by repeating samples, so at most one study per rank is counted\ntwice out of ~870 -- below the resolution of the measurement.\n\n**The effective batch is held constant.** Each rank receives `batch_size`\nstudies, so the effective batch is `batch_size x devices x accumulation`.\nHalving accumulation when moving from one device to two keeps the optimiser\nseeing the same eight studies per step, and keeps the OneCycleLR step count\nunchanged.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom src import config\nfrom src.model import dataset as knee_dataset\nfrom src.model import net\n\ntry:  # Lightning 2.x ships under both names depending on the install.\n    import lightning.pytorch as pl\nexcept ImportError:  # pragma: no cover - exercised only on Kaggle/Colab\n    import pytorch_lightning as pl\n\n\ndef auc(y_true: np.ndarray, scores: np.ndarray) -> float:\n    """ROC AUC via the rank statistic, with ties averaged.\n\n    Duplicated from `labels.calibrate` rather than imported: only the model\n    package is materialised into the training notebook, and a metric that\n    disagrees with the one the labels were scored by would be worse than the\n    duplication.\n    """\n    y_true = np.asarray(y_true)\n    scores = np.asarray(scores, dtype=float)\n    positives = y_true == 1\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float("nan")\n\n    order = np.argsort(scores, kind="mergesort")\n    ranks = np.empty(len(scores), dtype=float)\n    ranks[order] = np.arange(1, len(scores) + 1)\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\n                                        return_counts=True)\n    sums = np.zeros(len(unique))\n    np.add.at(sums, inverse, ranks)\n    ranks = (sums / counts)[inverse]\n\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\n\n\ndef macro_auc(truth: np.ndarray, probs: np.ndarray) -> float:\n    """Mean per-finding AUC, ignoring findings with no positives present."""\n    scores = [auc(truth[:, i], probs[:, i]) for i in range(truth.shape[1])]\n    return float(np.nanmean(scores))\n\n\nclass StudyVolumeDataset(Dataset):\n    """Study volumes with an even slice subsample and intensity jitter.\n\n    Slices are subsampled evenly rather than at random so a validation score\n    is reproducible. Horizontal flips are deliberately not used: after\n    canonicalisation a flip re-mirrors the knee and swaps medial with lateral,\n    silently corrupting four of the twelve labels.\n    """\n\n    def __init__(self, study_index, studies: Sequence[str],\n                 cache_paths: Sequence[str], slices_per_series: int,\n                 augment: bool = False):\n        self.index = study_index\n        self.studies = list(studies)\n        self.cache_paths = list(cache_paths)\n        self.slices_per_series = slices_per_series\n        self.augment = augment\n        self.keep = np.linspace(0, config.CACHE_SLICES - 1,\n                                slices_per_series).round().astype(int)\n        self._shards: Optional[Dict[str, np.ndarray]] = None\n\n    @property\n    def shards(self) -> Dict[str, np.ndarray]:\n        """Open the memmaps on first use, once per process."""\n        if self._shards is None:\n            self._shards = {path: np.load(path, mmap_mode="r")\n                            for path in self.cache_paths}\n        return self._shards\n\n    def __getstate__(self):\n        # Never serialise the memmaps into a spawned process or a dataloader\n        # worker; each one reopens them from the paths instead.\n        state = self.__dict__.copy()\n        state["_shards"] = None\n        return state\n\n    def __len__(self) -> int:\n        return len(self.studies)\n\n    def __getitem__(self, i: int):\n        item = knee_dataset.load_study(self.index, self.studies[i], self.shards)\n\n        available = item["volumes"].shape[1]\n        if available > self.slices_per_series:\n            # A thinned cache is already at the target count; only subsample a\n            # full one, or the second cut would drop slices twice.\n            item["volumes"] = item["volumes"][:, self.keep]\n        elif available < self.slices_per_series:\n            # The attention pool masks over the slice axis, so a short volume\n            # trains happily on fewer slices than configured and says nothing.\n            raise ValueError(\n                f"cache supplies {available} slices, slices_per_series is "\n                f"{self.slices_per_series} -- attach the full 24-slice "\n                "shards, or lower the setting")\n\n        if self.augment:\n            # Intensity jitter only. A geometric flip would swap medial and\n            # lateral; rotation is safe but not part of this measurement.\n            scale = np.random.uniform(0.9, 1.1)\n            item["volumes"] = np.clip(\n                item["volumes"].astype(np.float32) * scale,\n                0, 255).astype(np.uint8)\n        return item\n\n\nclass KneeLitModule(pl.LightningModule):\n    """The fine-tuning step, loss, schedule, and epoch-level scoring.\n\n    Two validation dataloaders are expected, and which is which matters:\n    index 0 is the validation fold scored against mined labels binarised at\n    0.5, and it is the only thing allowed to select a checkpoint. Index 1 is\n    the gold 58, logged every epoch because it is nearly free and shows\n    whether the run is converging -- and never monitored. Selecting on gold\n    turns the project\'s one honest measuring stick into a model-selection set;\n    an earlier version of this loop did exactly that and inflated its own\n    reported figure by an estimated 0.01-0.03.\n    """\n\n    def __init__(self, backbone: torch.nn.Module, feature_dim: int,\n                 lr: float = 1e-4, weight_decay: float = 1e-4,\n                 total_steps: int = 1000, max_lr_multiplier: float = 3.0,\n                 report_dim: Optional[int] = None,\n                 lambda_report: float = 0.0, lambda_rank: float = 0.0,\n                 rank_queue_size: int = 256, rank_min_gap: float = 0.25,\n                 aux_mode: str = "fixed", aux_gate: str = "binary"):\n        super().__init__()\n        if aux_mode not in ("fixed", "cosine"):\n            raise ValueError(f"aux_mode must be fixed|cosine, got {aux_mode!r}")\n        if aux_gate not in ("binary", "weighted"):\n            raise ValueError(f"aux_gate must be binary|weighted, got {aux_gate!r}")\n        self.save_hyperparameters(ignore=["backbone"])\n        self.model = net.KneeNet(backbone, feature_dim)\n        self.lr = lr\n        self.weight_decay = weight_decay\n        self.total_steps = total_steps\n        self.max_lr_multiplier = max_lr_multiplier\n        self.lambda_report = lambda_report\n        self.lambda_rank = lambda_rank\n        self.rank_queue_size = rank_queue_size\n        self.rank_min_gap = rank_min_gap\n        self.aux_mode = aux_mode\n        self.aux_gate = aux_gate\n\n        # The report head is an attribute of the *wrapper*, deliberately: the\n        # exported state dict keeps only `model.`-prefixed keys, so auxiliary\n        # parameters vanish at export and notebook 10 keeps loading a bare\n        # KneeNet. Created only when actually trained against -- under DDP a\n        # parameter that never receives gradient is an error, not a no-op.\n        if report_dim is not None and lambda_report > 0:\n            self.report_head = torch.nn.Sequential(\n                torch.nn.LayerNorm(feature_dim),\n                torch.nn.Linear(feature_dim, report_dim))\n        else:\n            self.report_head = None\n\n        # Cross-batch queue for the ranking loss. At one study per rank there\n        # are no pairs inside a batch, so current logits are ranked against\n        # detached logits from recent steps (XBM-style). Slightly stale scores\n        # are the price; per-rank and reset each epoch.\n        self._rank_queue: List[Dict[str, torch.Tensor]] = []\n\n        # Not `_buffers`: nn.Module reserves that name for registered\n        # buffers, and shadowing it corrupts state_dict and .to(device).\n        self._val_outputs: Dict[int, List[Dict[str, torch.Tensor]]] = {0: [], 1: []}\n\n    def forward(self, volumes, meta, mask):\n        return self.model(volumes, meta, mask)\n\n    def on_train_epoch_start(self):\n        self._rank_queue = []\n\n    def _report_loss(self, pooled: torch.Tensor, batch) -> torch.Tensor:\n        """Cosine distance between the projected study embedding and the\n        report\'s sentence embedding, over studies that have one.\n\n        The head must run even when the whole batch lacks reports: DDP\n        requires every parameter to join every backward pass.\n        """\n        predicted = self.report_head(pooled.float())\n        if "report" not in batch:\n            return predicted.sum() * 0.0\n        predicted = torch.nn.functional.normalize(predicted, dim=-1)\n        target = torch.nn.functional.normalize(batch["report"].float(), dim=-1)\n        distance = 1.0 - (predicted * target).sum(dim=-1)\n        mask = batch["report_mask"].float()\n        return (distance * mask).sum() / mask.sum().clamp(min=1.0)\n\n    def _rank_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\n        scores = logits.float()\n        targets = batch["targets"].float()\n        weights = batch["weights"].float()\n\n        if self._rank_queue:\n            loss = net.pairwise_rank_loss(\n                scores, targets,\n                torch.cat([q["scores"] for q in self._rank_queue]),\n                torch.cat([q["targets"] for q in self._rank_queue]),\n                weights,\n                torch.cat([q["weights"] for q in self._rank_queue]),\n                min_gap=self.rank_min_gap)\n        else:\n            loss = scores.sum() * 0.0\n\n        self._rank_queue.append({"scores": scores.detach(),\n                                 "targets": targets.detach(),\n                                 "weights": weights.detach()})\n        while sum(len(q["scores"]) for q in self._rank_queue) > self.rank_queue_size:\n            self._rank_queue.pop(0)\n        return loss\n\n    def _aux_cosine(self, main_loss: torch.Tensor, aux_loss: torch.Tensor,\n                    pooled: torch.Tensor) -> torch.Tensor:\n        """cos between the two losses\' gradients at the shared bottleneck.\n\n        Du et al., "Adapting Auxiliary Losses Using Gradient Similarity"\n        (arXiv:1812.02224), Algorithm 1: apply the auxiliary gradient to the\n        shared parameters only while it points into the same half-space as\n        the main gradient, which guarantees convergence to the main task\'s\n        critical points and blocks negative transfer.\n\n        The paper measures that cosine over the shared parameters theta.\n        Here every shared parameter is reached *only* through `pooled` --\n        `forward_pooled` returns `head(pooled), pooled`, and both losses hang\n        off that one tensor -- so the shared gradients are J^T a and J^T b for\n        the same Jacobian J = d pooled/d theta. Taking the cosine at the\n        bottleneck asks the same question ("does the auxiliary want to move\n        the shared representation the way the main task does?") for the price\n        of two backward passes through the two small heads instead of two\n        through ConvNeXt. It is a proxy, not an identity: J^T can rotate the\n        pair, so agreement here is not arithmetically agreement at theta.\n        The sign is what Algorithm 1 consumes, and the paper itself keeps\n        only the sign because minibatch cosines are too noisy to trust\n        beyond it.\n\n        retain_graph is required: the real backward still has to run\n        afterwards over the same graph.\n\n        Both losses are scaled by a constant first. Training runs in\n        `16-mixed`, and these probe gradients bypass Lightning\'s GradScaler,\n        so an unscaled fp16 backward can flush small values to zero -- which\n        reads as cosine 0, i.e. "aligned", i.e. silently back to a fixed\n        lambda. Cosine is scale-invariant, so scaling both sides costs\n        nothing and keeps the values off the floor.\n        """\n        scale = 1024.0\n        grad_main, = torch.autograd.grad(main_loss * scale, pooled,\n                                         retain_graph=True)\n        grad_aux, = torch.autograd.grad(aux_loss * scale, pooled,\n                                        retain_graph=True)\n        cosine = torch.nn.functional.cosine_similarity(\n            grad_main.flatten().float(), grad_aux.flatten().float(), dim=0)\n        # Ranks must agree, or the "binary" gate is silently a soft one: DDP\n        # averages the two ranks\' gradients, so a per-rank decision would\n        # apply the auxiliary at half strength whenever they disagreed.\n        # torch.distributed rather than self.all_gather, which requires an\n        # attached Trainer and would make this untestable in isolation.\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\n            cosine = cosine.clone()\n            torch.distributed.all_reduce(cosine, op=torch.distributed.ReduceOp.SUM)\n            cosine = cosine / torch.distributed.get_world_size()\n        return cosine\n\n    def _aux_weight(self, cosine: torch.Tensor) -> torch.Tensor:\n        """Algorithm 1 (binary) or Algorithm 2 (cosine-weighted)."""\n        if self.aux_gate == "weighted":\n            return cosine.clamp(min=0.0)\n        return (cosine >= 0).to(cosine.dtype)\n\n    def training_step(self, batch, batch_idx):\n        logits, pooled = self.model.forward_pooled(\n            batch["volumes"], batch["meta"], batch["mask"])\n        loss = net.weighted_bce(logits, batch["targets"], batch["weights"])\n        self.log("bce_loss", loss, on_step=False, on_epoch=True,\n                 sync_dist=True, batch_size=len(logits))\n        main_loss = loss\n\n        if self.report_head is not None:\n            report_loss = self._report_loss(pooled, batch)\n            weight = self.lambda_report\n            if self.aux_mode == "cosine":\n                cosine = self._aux_cosine(main_loss, report_loss, pooled)\n                weight = weight * self._aux_weight(cosine)\n                # The diagnostic worth having even when the gate never fires:\n                # a report loss that is mostly anti-aligned explains a fixed\n                # lambda hurting the leaderboard while helping validation.\n                self.log("report_cosine", cosine, on_step=False, on_epoch=True,\n                         sync_dist=True, batch_size=len(logits))\n                self.log("report_gate", (cosine >= 0).float(), on_step=False,\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\n            loss = loss + weight * report_loss\n            self.log("report_loss", report_loss, on_step=False, on_epoch=True,\n                     sync_dist=True, batch_size=len(logits))\n\n        if self.lambda_rank > 0:\n            rank_loss = self._rank_loss(logits, batch)\n            weight = self.lambda_rank\n            if self.aux_mode == "cosine":\n                cosine = self._aux_cosine(main_loss, rank_loss, pooled)\n                weight = weight * self._aux_weight(cosine)\n                self.log("rank_cosine", cosine, on_step=False, on_epoch=True,\n                         sync_dist=True, batch_size=len(logits))\n            loss = loss + weight * rank_loss\n            self.log("rank_loss", rank_loss, on_step=False, on_epoch=True,\n                     sync_dist=True, batch_size=len(logits))\n\n        self.log("train_loss", loss, prog_bar=True, on_step=True,\n                 on_epoch=True, sync_dist=True, batch_size=len(logits))\n        return loss\n\n    def validation_step(self, batch, batch_idx, dataloader_idx=0):\n        logits = self(batch["volumes"], batch["meta"], batch["mask"])\n        self._val_outputs[dataloader_idx].append({\n            "probs": torch.sigmoid(logits.float()),\n            "targets": batch["targets"].float(),\n        })\n\n    def _gathered(self, dataloader_idx: int):\n        """All-gather one buffer and return (truth, probs) as numpy."""\n        entries = self._val_outputs[dataloader_idx]\n        if not entries:\n            return None, None\n        probs = torch.cat([e["probs"] for e in entries])\n        targets = torch.cat([e["targets"] for e in entries])\n\n        # all_gather returns (world_size, N, L) when every rank holds the same\n        # N, which the distributed sampler guarantees by padding.\n        probs = self.all_gather(probs)\n        targets = self.all_gather(targets)\n        if probs.ndim == 3:\n            probs = probs.reshape(-1, probs.shape[-1])\n            targets = targets.reshape(-1, targets.shape[-1])\n        return (targets.cpu().numpy(), probs.cpu().numpy())\n\n    def on_validation_epoch_end(self):\n        val_truth, val_probs = self._gathered(0)\n        if val_truth is not None:\n            # Mined labels are soft; binarise for a ranking metric.\n            val_macro = macro_auc((val_truth >= 0.5).astype(int), val_probs)\n            self.log("val_macro", val_macro, prog_bar=True, rank_zero_only=False)\n\n        gold_truth, gold_probs = self._gathered(1)\n        if gold_truth is not None:\n            gold_macro = macro_auc(gold_truth.astype(int), gold_probs)\n            # Monitoring only. Never pass this to a checkpoint callback.\n            self.log("gold_macro", gold_macro, prog_bar=True,\n                     rank_zero_only=False)\n\n        self._val_outputs = {0: [], 1: []}\n\n    def configure_optimizers(self):\n        optimiser = torch.optim.AdamW(self.parameters(), lr=self.lr,\n                                      weight_decay=self.weight_decay)\n        scheduler = torch.optim.lr_scheduler.OneCycleLR(\n            optimiser, max_lr=self.lr * self.max_lr_multiplier,\n            total_steps=self.total_steps)\n        return {\n            "optimizer": optimiser,\n            # Stepped per optimiser step, so accumulation is already folded in.\n            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},\n        }\n\n\ndef optimiser_steps(n_studies: int, batch_size: int, devices: int,\n                    accumulate: int, epochs: int) -> int:\n    """Total OneCycleLR steps, which must match what Lightning will take.\n\n    A schedule sized for the wrong number of steps either finishes its cycle\n    early and trains the tail at a near-zero learning rate, or never reaches\n    its peak. Neither raises.\n    """\n    per_rank_batches = int(np.ceil(n_studies / (batch_size * max(devices, 1))))\n    return epochs * (per_rank_batches // accumulate + 1)\n\n\nclass EpochReport(pl.Callback):\n    """One line per epoch, from rank zero only.\n\n    Under DDP every rank runs this code, so an unguarded print produces one\n    copy of the log per GPU and makes a run look twice as long as it is.\n\n    **This reads `callback_metrics` in `on_validation_end`, not in\n    `on_validation_epoch_end`.** Lightning runs callback hooks before the\n    LightningModule\'s hook of the same name, so a callback asking for\n    `val_macro` at `on_validation_epoch_end` arrives before\n    `KneeLitModule.on_validation_epoch_end` has computed it and reads an\n    empty dict -- printing `nan` for every epoch of an otherwise healthy run.\n    `ModelCheckpoint` selects in `on_validation_end` for the same reason, so\n    checkpoint selection was never affected by this; only the log was.\n    """\n\n    def __init__(self):\n        self.rows: List[Dict[str, float]] = []\n\n    def on_validation_end(self, trainer, module):\n        if trainer.sanity_checking or not trainer.is_global_zero:\n            return\n        metrics = trainer.callback_metrics\n        val = float(metrics.get("val_macro", float("nan")))\n        gold = float(metrics.get("gold_macro", float("nan")))\n        self.rows.append({"epoch": trainer.current_epoch + 1,\n                          "val_macro": val, "gold_macro": gold})\n        best = max(row["val_macro"] for row in self.rows)\n        marker = "  <- best" if val >= best else ""\n        print(f"  epoch {trainer.current_epoch + 1}: val macro {val:.3f}  "\n              f"gold {gold:.3f} (monitor only){marker}")\n\n\nclass BatchProgress(pl.Callback):\n    """Plain-text tqdm over training batches, rank zero only.\n\n    Lightning\'s own progress bar resolves to the notebook-widget flavour of\n    tqdm inside Kaggle, and the widget never reaches the log of a\n    Save & Run All -- eleven minutes pass per epoch with no sign of life.\n    Plain tqdm writes carriage-returned text to stdout, which survives both\n    the interactive view and the batch log, so the Trainer disables its own\n    bar and takes this instead. `mininterval` keeps the batch log from\n    bloating: one refresh per interval, not one per batch.\n    """\n\n    def __init__(self, refresh_seconds: float = 15.0):\n        self.refresh_seconds = refresh_seconds\n        self._bar = None\n\n    def on_train_epoch_start(self, trainer, module):\n        if not trainer.is_global_zero:\n            return\n        import sys\n        from tqdm import tqdm\n        self._bar = tqdm(\n            total=int(trainer.num_training_batches),\n            desc=f"epoch {trainer.current_epoch + 1}/{trainer.max_epochs}",\n            unit="batch", file=sys.stdout,\n            mininterval=self.refresh_seconds, leave=False)\n\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\n        if self._bar is not None:\n            self._bar.update(1)\n\n    def on_train_epoch_end(self, trainer, module):\n        if self._bar is not None:\n            self._bar.close()\n            self._bar = None\n\n\ndef export_from_checkpoint(ckpt_path: str, out_path: str) -> int:\n    """Convert a Lightning checkpoint into the plain state dict for scoring.\n\n    Read from the file rather than from a module in memory: under the spawn\n    strategy the process that called `fit` is not the process that trained,\n    and reaching for in-memory weights there is how a run silently ships its\n    initialisation. Returns the number of tensors written.\n    """\n    checkpoint = torch.load(ckpt_path, map_location="cpu")\n    state_dict = checkpoint.get("state_dict", checkpoint)\n    state = {key[len("model."):]: value\n             for key, value in state_dict.items()\n             if key.startswith("model.")}\n    if not state:\n        raise ValueError(\n            f"no \'model.\' keys in {ckpt_path}; found "\n            f"{list(state_dict)[:3]}")\n    torch.save(state, out_path)\n    return len(state)\n\n\ndef export_state_dict(module: KneeLitModule, path: str) -> None:\n    """Write a plain `KneeNet` state dict, the format the submission loads.\n\n    Lightning prefixes every key with `model.` because `KneeNet` is an\n    attribute of the wrapper. Notebook 10 constructs a bare `KneeNet` and\n    calls `load_state_dict`, so the prefix has to come off here rather than\n    become a surprise at submission time.\n    """\n    state = {key[len("model."):]: value\n             for key, value in module.state_dict().items()\n             if key.startswith("model.")}\n    torch.save(state, path)\n'
}

for path, source in FILES.items():
    target = pathlib.Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)

sys.path.insert(0, ".")

# Drop any previously imported copy before importing. Writing the files above
# does not affect a module already in sys.modules, so re-running this cell in a
# kernel that has already imported src/ silently keeps the OLD code -- the new
# file sits on disk unused, and the run looks fine until something calls a
# function that only exists in the new version.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

from src import config
from src.data import dicom_io, cache, index as index_mod

# Only some notebooks materialise the model package.
if pathlib.Path("src/model/side_classifier.py").exists():
    from src.model import side_classifier
if pathlib.Path("src/model/net.py").exists():
    from src.model import embed, dataset as knee_dataset, net, splits
# Only the fine-tuning notebook materialises this one; it imports lightning at
# module scope, which nothing else needs.
if pathlib.Path("src/model/lightning.py").exists():
    from src.model import lightning as knee_lightning

print("modules ready:", len(FILES), "files")
print("labels:", len(config.LABELS))
print("cache geometry:", config.CACHE_SLICES, "x", config.CACHE_SIZE)
print("data root:", config.DATA_ROOT)

## 3. Assemble

One shard only. ~1,460 studies is enough to measure throughput and train a
first fold, and loading all three would cost memory without changing what this
run establishes.

In [ ]:
import glob

import numpy as np
import pandas as pd


def find_inputs(pattern):
    # ./data first, so the same notebook runs on Kaggle with the cache
    # attached and on Colab with it downloaded by notebook 09.
    found = sorted(glob.glob(f"data/**/{pattern}", recursive=True))
    found += sorted(glob.glob(f"/kaggle/working/{pattern}"))
    for root in sorted(glob.glob("/kaggle/input/*")):
        if os.path.basename(root) == "competitions":
            continue
        found += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    return found


def find_competition_file(name):
    for candidate in (f"data/{name}",
                      f"/kaggle/input/competitions/*/{name}",
                      f"/kaggle/input/*/{name}"):
        hits = sorted(glob.glob(candidate))
        if hits:
            return hits[0]
    raise SystemExit(f"missing competition file: {name}")


labels = pd.read_parquet(find_inputs("train_labels.parquet")[0])
sides = pd.read_parquet(find_inputs("side_predictions.parquet")[0])

# Report sentence embeddings, built locally by src/labels/report_embed.py and
# uploaded alongside the labels. Optional: without the file the auxiliary
# report loss is disabled and training proceeds as before.
report_hits = find_inputs("report_embeddings.parquet")
reports = pd.read_parquet(report_hits[0]) if report_hits else None
if reports is None:
    print("report_embeddings.parquet not attached -- report loss disabled")
series_meta = pd.read_csv(find_competition_file("train_series.csv"))
train_csv = pd.read_csv(find_competition_file("train.csv"))
gold = train_csv[train_csv["ACL"].notna()].reset_index(drop=True)

# The measurement run is done: 2.5 min/epoch on one shard, ~7.6 min on all
# three, and the gold score was still climbing at epoch 4 rather than
# plateauing. So this is the real run.
ALL_SHARDS = True

cache_paths = find_inputs("cache_train_series_*.npy")
if not ALL_SHARDS:
    cache_paths = cache_paths[:1]

shards, frames = {}, []
for cache_path in cache_paths:
    manifest_path = cache_path.replace("cache_", "manifest_").replace(".npy", ".parquet")
    if not os.path.exists(manifest_path):
        stem = os.path.basename(cache_path).replace("cache_", "").replace(".npy", "")
        hits = [m for m in find_inputs("manifest_train_series_*.parquet")
                if os.path.basename(m).startswith(f"manifest_{stem}")]
        if not hits:
            continue
        manifest_path = hits[0]
    frame = pd.read_parquet(manifest_path)
    frame["cache_path"] = cache_path
    shards[cache_path] = np.load(cache_path, mmap_mode="r")
    frames.append(frame)
    print("shard:", os.path.basename(cache_path), shards[cache_path].shape)

manifest = pd.concat(frames, ignore_index=True)

folded = splits.make_folds(labels)
splits.assert_no_gold_leakage(folded["StudyInstanceUID"], set(gold["StudyInstanceUID"]))

# Capacity ladder, step 2. Doubling slices was worth +0.024 on the
# leaderboard, so the other axis the study is subsampled on is the next thing
# to test. Most studies hold ~5.5 series and three were being kept; PF OA is
# one of only two findings where this model still loses to frozen DINOv2
# features, and the patella may simply not be in the three that survive.
MAX_SERIES = 5

index = knee_dataset.StudyIndex(manifest, series_meta, folded, sides,
                                max_series=MAX_SERIES, reports=reports)
gold_labels = gold[["StudyInstanceUID"] + config.LABELS].copy()
for name in config.LABELS:
    gold_labels[f"{name}_weight"] = 1.0
gold_index = knee_dataset.StudyIndex(manifest, series_meta, gold_labels, sides,
                                     max_series=MAX_SERIES)
print(f"studies in this shard: {len(index)}  gold present: {len(gold_index)}")

## 4. Dataset

`StudyVolumeDataset` lives in `src/model/lightning.py` rather than in this
cell. Lightning's notebook DDP strategy spawns a process per GPU and pickles
the dataset into each one; a class defined in a notebook cell belongs to
`__main__` and does not survive that. The same rule already sent the cache's
process-pool worker into `src/data/cache.py`.

The memory-mapped shards are deliberately **not** handed to the dataset. It
takes paths and opens them lazily per process, and drops the handles in
`__getstate__`, so ~29 GB of memmap never gets serialised into a worker.

In [ ]:
from torch.utils.data import DataLoader

# The capacity ladder is finished: 8 -> 16 slices paid +0.024 on the
# leaderboard, 3 -> 5 series +0.020, and 16 -> 24 slices +0.002 while the
# gold curve began falling after epoch 7 -- overfitting, not headroom.
#
# Experiments therefore iterate at 16 slices, the reference geometry behind
# LB 0.889 / gold 0.861 on fold 1. The current experiment is the pair of
# auxiliary losses above; 24 remains a one-line change reserved for a final
# AUC-slot run, where its +0.002 may still be worth the runtime.
SLICES_PER_SERIES = 16

# One study per rank, held over from the five-series rung (1 x 5 x 16 = 80
# images per step). If it OOMs the fallback is MAX_SERIES 5 -> 4, not a
# lower slice count, because slices are the axis that has paid twice.
BATCH_STUDIES = 1

# Effective batch is BATCH_STUDIES x DEVICES x ACCUM_STEPS, and it must stay
# at eight studies whether this runs on one GPU or two -- otherwise the two
# configurations are not comparable and the OneCycleLR step count shifts
# underneath the schedule.
EFFECTIVE_STUDIES = 8
ACCUM_STEPS = max(1, EFFECTIVE_STUDIES // (BATCH_STUDIES * DEVICES))
print(f"{BATCH_STUDIES} studies x {DEVICES} devices x {ACCUM_STEPS} accum "
      f"= {BATCH_STUDIES * DEVICES * ACCUM_STEPS} effective")

CACHE_PATHS = list(shards)


def make_loader(study_index, studies, shuffle=False, augment=False):
    return DataLoader(
        knee_lightning.StudyVolumeDataset(
            study_index, studies, CACHE_PATHS, SLICES_PER_SERIES, augment),
        batch_size=BATCH_STUDIES, shuffle=shuffle, num_workers=2,
        collate_fn=knee_dataset.collate_studies, pin_memory=True,
        persistent_workers=True)

## 5. Model

ConvNeXt-T with ImageNet weights, which the competition permits. The
aggregation above it is the same `KneeNet` the frozen path uses, so the
comparison is between backbones rather than between architectures.

In [ ]:
import timm

BACKBONE = "convnext_tiny.fb_in22k_ft_in1k"

# Built once here before the trainer spawns its workers. Each rank constructs
# its own backbone, and two processes racing to download the same weights into
# the same cache is a failure that only appears on multi-GPU runs -- warming
# the cache first turns that race into two cache hits.
_warm = timm.create_model(BACKBONE, pretrained=True, num_classes=0, in_chans=3)
FEATURE_DIM = _warm.num_features
print(f"{BACKBONE}: {FEATURE_DIM} features, "
      f"{sum(p.numel() for p in _warm.parameters()) / 1e6:.1f}M params")
del _warm

auc = knee_lightning.auc

## 6. Train one fold

`FOLD` below decides which of the five folds this session trains and validates
on; five sessions with `FOLD = 0` through `4` produce the checkpoints notebook
10 ensembles. The best epoch is chosen on the **validation fold** against the
mined labels. Gold is printed every epoch but never selects anything -- an
earlier version checkpointed on gold macro, which quietly turned the only
honest measuring stick into a model-selection set and inflated its reading by
an estimated 0.01-0.03.

In [ ]:
import time

import timm

# The measurement run reached 0.729 at epoch 4 and was still rising
# (0.584 -> 0.679 -> 0.708 -> 0.729), so four was undertrained.
EPOCHS = 12
LR = 1e-4

# Two auxiliary objectives, both training-time only and both dropped at
# export, so notebook 10 is untouched. Either is disabled by setting its
# weight to 0.0, which is how a single-variable ablation is run.
#
# LAMBDA_REPORT: the pooled study embedding regresses the report's sentence
# embedding (cosine distance). The report describes more than twelve numbers,
# and this hands that richer summary to the network as extra supervision --
# also a lever against the overfitting the 24-slice run showed, since the
# regression target is far harder to memorise than twelve soft labels.
#
# LAMBDA_RANK: the metric is macro AUC, a ranking statistic, but BCE
# optimises calibration. This RankNet term orders the current study against
# a queue of recent (detached) predictions, pairing only studies whose soft
# targets differ by >= 0.25 so extraction noise is not what gets ranked.
# Screen result (fold 1, both at 0.2): gold 0.854 vs the 0.861 reference.
# Both losses trained (report 0.17->0.044, rank 0.65->0.026), but rank
# collapsed to near-zero -- memorised its own queue.
#
# LB verdict (2026-08-26): report 0.5 alone scored val 0.881 / gold 0.867
# on fold 1 at 16 x 5 -- the best fold-1 numbers measured -- but the
# submission came in at LB 0.880 against the aux-free twin's 0.889 on the
# identical geometry. Val labels are mined from the same reports the aux
# target embeds, so the val gain was partly circular, and gold's +0.006 sat
# inside its noise floor. Both lambdas therefore ship at 0.0; the code
# stays for any later phase that reaches for more capacity.
LAMBDA_REPORT = 1.0 if reports is not None else 0.0
LAMBDA_RANK = 0.0

# AUX_MODE selects how the lambdas above are applied.
#
#   "fixed"  -- the weight is the constant above, every step. What C2 ran.
#   "cosine" -- Du et al., "Adapting Auxiliary Losses Using Gradient
#               Similarity" (arXiv:1812.02224), Algorithm 1: apply the
#               auxiliary gradient only while its cosine against the main
#               gradient is non-negative, i.e. only while it is a descent
#               direction for the main task. The paper's whole argument is
#               that a *constant* lambda is the bug -- an auxiliary task's
#               usefulness changes during training, so a fixed weight keeps
#               pushing after it has started to hurt. Set LAMBDA_REPORT = 1.0
#               with this mode to reproduce the paper exactly; the lambda
#               then scales the auxiliary only on the steps it is switched on.
#
# Even with the lambdas at 0 this changes nothing, so the fold ensemble is
# unaffected. The run to make first is the diagnostic one: LAMBDA_REPORT = 1.0
# and AUX_MODE = "cosine" logs `report_cosine` and `report_gate` per epoch,
# which say whether C2's report loss was ever aligned with the main task.
AUX_MODE = "cosine"
AUX_GATE = "binary"   # "weighted" uses max(0, cos) instead of an on/off gate
REPORT_DIM = (len([c for c in reports.columns if c != "StudyInstanceUID"])
              if reports is not None else None)

# Which fold this session trains.
#
# THE FOLD AND THE AUX SETTINGS MOVE TOGETHER. This notebook serves two runs
# and they are mutually exclusive:
#
#   Ensemble fold   FOLD = 0 | 2 | 3 | 4, LAMBDA_REPORT = 0.0,
#                   AUX_MODE = "fixed". Fold 1 already exists (16 x 5
#                   reference, LB 0.889), so the sweep is the other four.
#   Aux experiment  FOLD = 1, LAMBDA_REPORT = 1.0, AUX_MODE = "cosine".
#                   Fold 1 because it is the ONLY split with a measured
#                   baseline (val 0.872 / gold 0.861 / LB 0.889); the same
#                   run on any other fold has nothing to be compared against.
#
# Changing the fold and the configuration together has already cost this
# project one ambiguous run, and an ensemble fold trained with an auxiliary
# loss is worse than useless -- it silently makes one member of the average
# a different model. The banner printed below states which run this is.
FOLD = 1
CHECKPOINT = f"/kaggle/working/convnext_fold{FOLD}.pt"

keyed = folded.set_index("StudyInstanceUID")
train_studies = [s for s in index.studies if keyed.loc[s, "fold"] != FOLD]
val_studies = [s for s in index.studies if keyed.loc[s, "fold"] == FOLD]

train_loader = make_loader(index, train_studies, shuffle=True, augment=True)
# Dataloader 0 is the validation fold, carrying mined soft targets -- the only
# thing allowed to select. Dataloader 1 is gold, carrying real 0/1 labels,
# logged for convergence and never monitored.
val_loader = make_loader(index, val_studies)
gold_loader = make_loader(gold_index, gold_index.studies)
print(f"fold {FOLD}: train {len(train_studies)}  val {len(val_studies)}  "
      f"gold {len(gold_index)}")

TOTAL_STEPS = knee_lightning.optimiser_steps(
    len(train_studies), BATCH_STUDIES, DEVICES, ACCUM_STEPS, EPOCHS)
print("optimiser steps:", TOTAL_STEPS)

module = knee_lightning.KneeLitModule(
    backbone=timm.create_model(BACKBONE, pretrained=True, num_classes=0,
                               in_chans=3),
    feature_dim=FEATURE_DIM, lr=LR, total_steps=TOTAL_STEPS,
    report_dim=REPORT_DIM, lambda_report=LAMBDA_REPORT,
    lambda_rank=LAMBDA_RANK, aux_mode=AUX_MODE, aux_gate=AUX_GATE)
print(f"aux losses: report {LAMBDA_REPORT} (dim {REPORT_DIM})  "
      f"rank {LAMBDA_RANK}  mode {AUX_MODE}"
      + (f"/{AUX_GATE}" if AUX_MODE == "cosine" else ""))

# Which of the notebook's two mutually exclusive runs this is. An ensemble
# fold trained with an auxiliary loss makes one member of the average a
# different model, and the averaging hides it -- so say it before the hours
# are spent, not after the leaderboard disagrees.
_aux_on = LAMBDA_REPORT > 0 or LAMBDA_RANK > 0
print("=" * 68)
if _aux_on and FOLD == 1:
    print("  AUX EXPERIMENT: fold 1, the only split with a baseline")
    print("  compare against val 0.872 / gold 0.861 / LB 0.889")
    print("  read report_gate in metrics.csv; decide on the LB, not on val")
elif not _aux_on:
    print(f"  ENSEMBLE FOLD {FOLD}: reference config, no auxiliary loss")
    print("  export convnext_fold{}.pt and keep it for the 5-fold run".format(FOLD))
else:
    print(f"  MISCONFIGURED: auxiliary loss on with FOLD = {FOLD}.")
    print("  This is neither an ensemble fold (which needs the lambdas at")
    print("  0.0) nor a readable experiment (which needs fold 1). Stop and")
    print("  set one of the two configurations at the top of this notebook.")
print("=" * 68)

checkpointer = pl.callbacks.ModelCheckpoint(
    dirpath="/kaggle/working", filename=f"lit_fold{FOLD}",
    monitor="val_macro", mode="max", save_top_k=1)
report = knee_lightning.EpochReport()

# Metrics go to a CSV as well as to stdout. Under ddp_notebook every rank is
# a forked child, and a child's prints do not always find their way back to
# the cell output -- so the per-epoch numbers are read back from disk below,
# in this process, where they are guaranteed to be visible.
csv_logger = pl.loggers.CSVLogger("/kaggle/working", name=f"lit_fold{FOLD}",
                                  version="metrics")

# ddp_notebook forks rather than re-executing the script, which is what makes
# DDP usable from a notebook at all -- plain "ddp" deadlocks here. If the fork
# route ever proves unworkable, "ddp_spawn" is a drop-in: everything the child
# needs is picklable by design, since the dataset and module live in
# src/model/lightning.py and the memmaps are dropped in __getstate__.
STRATEGY = "ddp_notebook" if DEVICES > 1 else "auto"

# The fork cannot survive a CUDA context that already exists here. Catch it
# now, with an instruction, rather than as a sixty-line traceback out of a
# child process after the model and loaders are built.
if STRATEGY == "ddp_notebook" and torch.cuda.is_initialized():
    raise SystemExit(
        "CUDA is already initialised in this process, so the forked workers "
        "cannot start.\n"
        "Run -> Restart Session, then run the cells in order. If it recurs, "
        "set STRATEGY = 'ddp_spawn' in this cell."
    )

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=DEVICES,
    strategy=STRATEGY,
    precision="16-mixed",
    accumulate_grad_batches=ACCUM_STEPS,
    callbacks=[checkpointer, report, knee_lightning.BatchProgress()],
    logger=csv_logger,
    log_every_n_steps=50,
    enable_model_summary=False,
    enable_progress_bar=False,
)

start = time.time()
trainer.fit(module, train_dataloaders=train_loader,
            val_dataloaders=[val_loader, gold_loader])
elapsed = time.time() - start

# EXPORT FIRST. Ninety minutes of GPU time exist only as a checkpoint file at
# this point, and anything that runs before the export is a chance to lose it
# to an unrelated error -- which is exactly what a stray reporting bug did on
# the first two-GPU run. Reporting is cosmetic and comes after, wrapped.
# Under ddp_notebook the fit happens in forked children, and Lightning
# transfers only `best_model_path` back to this process -- `best_model_score`
# is left None, and their launcher carries a "TODO: pass also best score" to
# that effect. Formatting it here crashed a finished twelve-epoch run. So the
# path is used, the score comes from the CSV below, and the path itself is
# checked against the filesystem rather than trusted.
checkpoint_path = checkpointer.best_model_path or ""
if not checkpoint_path or not os.path.exists(checkpoint_path):
    found = sorted(glob.glob(f"/kaggle/working/lit_fold{FOLD}*.ckpt"))
    if not found:
        raise SystemExit(
            "training finished but no checkpoint was written -- check the "
            "epoch log for whether val_macro was ever computed")
    checkpoint_path = found[0]
print("\nbest checkpoint:", checkpoint_path)

# Convert Lightning's checkpoint into the bare KneeNet state dict notebook 10
# loads. Read from the file, not from `module`: the process that called fit is
# not the process that trained.
n_tensors = knee_lightning.export_from_checkpoint(checkpoint_path, CHECKPOINT)
print(f"wrote {CHECKPOINT} ({n_tensors} tensors)")

# Re-read the epoch table rather than trusting the forked ranks' stdout. The
# logger's version is pinned, because the child resolves it during fit and a
# parent asking afterwards would count the child's directory as taken and
# point one version too far along an empty path.
try:
    history = pd.read_csv(os.path.join(csv_logger.log_dir, "metrics.csv"))
    per_epoch_rows = (history.dropna(subset=["val_macro"])
                      .groupby("epoch")[["val_macro", "gold_macro"]].last())
    best_epoch = per_epoch_rows["val_macro"].idxmax()
    print(f"\n{'epoch':>5s} {'val macro':>9s} {'gold':>7s}")
    for epoch_no, row in per_epoch_rows.iterrows():
        marker = "  <- selected" if epoch_no == best_epoch else ""
        print(f"{epoch_no + 1:5d} {row['val_macro']:9.3f} "
              f"{row['gold_macro']:7.3f}{marker}")
    best = per_epoch_rows.loc[best_epoch]
    print(f"\nbest val macro {best['val_macro']:.3f} at epoch "
          f"{best_epoch + 1}; gold there {best['gold_macro']:.3f} "
          "(monitor only, never selected on)")
except Exception as exc:  # noqa: BLE001 - the checkpoint is already safe
    print(f"\ncould not read the epoch table ({exc}); "
          f"look under {csv_logger.save_dir} for metrics.csv")

per_epoch = elapsed / EPOCHS
share = len(index) / 4349
print(f"\nthis shard is {share:.0%} of the training set")
print(f"per epoch: {per_epoch / 60:.1f} min on {DEVICES} GPU(s)")
print(f"projected 5 folds x {EPOCHS} epochs: "
      f"{per_epoch / share * EPOCHS * 5 / 3600:.1f} h")

## 7. Score

Against the gold studies present in this shard -- roughly a third of the 58, so
this is noisier than the frozen path's estimate. Read it as a direction, not a
number.

In [ ]:
# Scored in this process from the exported checkpoint, which is the artefact
# the submission will actually load -- not from a module left over in memory.
# If these numbers disagree with the last epoch's logged gold, the export is
# wrong, and that is worth knowing here rather than after a submission.
device = "cuda"

# Prefer the exported state dict, but fall back to the Lightning checkpoint
# it came from. A recovered run -- one where the training cell died after fit
# and the export was re-run by hand -- can easily have the .ckpt and not the
# .pt, and this cell is the verification step, so it must not be the thing
# that refuses to run.
scoring_path = CHECKPOINT
if not os.path.exists(scoring_path):
    found = sorted(glob.glob(f"/kaggle/working/lit_fold{FOLD}*.ckpt"))
    if not found:
        raise SystemExit(f"neither {CHECKPOINT} nor a lit_fold{FOLD} "
                         "checkpoint is present")
    scoring_path = found[0]
    print(f"{CHECKPOINT} missing; scoring from {scoring_path}")

scoring_model = net.KneeNet(
    timm.create_model(BACKBONE, pretrained=False, num_classes=0, in_chans=3),
    FEATURE_DIM).to(device)
scoring_model.load_state_dict(net.load_knee_state(scoring_path,
                                                  map_location=device))
scoring_model.eval()

gold_ids, rows = [], []
with torch.no_grad(), torch.amp.autocast("cuda"):
    for batch in gold_loader:
        logits = scoring_model(batch["volumes"].to(device),
                               batch["meta"].to(device),
                               batch["mask"].to(device))
        rows.append(torch.sigmoid(logits.float()).cpu().numpy())
        gold_ids += batch[config.ID_COL]
gold_probs = np.concatenate(rows)
truth = gold.set_index("StudyInstanceUID").loc[gold_ids]

FROZEN = dict(zip(config.LABELS,
                  [0.728, 0.671, 0.698, 0.680, 0.895, 0.899,
                   0.772, 0.897, 0.746, 0.862, 0.711, 0.683]))

print(f"{'finding':18s} {'this run':>9s}  {'frozen DINOv2':>13s}")
print("-" * 44)
scores = []
for i, name in enumerate(config.LABELS):
    value = auc(truth[name].to_numpy().astype(int), gold_probs[:, i])
    scores.append(value)
    print(f"{name:18s} {value:9.3f}  {FROZEN[name]:13.3f}")
print("-" * 44)
print(f"{'macro':18s} {np.nanmean(scores):9.3f}  {np.mean(list(FROZEN.values())):13.3f}")

## Next

Two things decide what follows.

**The runtime projection.** If five folds fit in the weekly GPU quota, run them
and compare against the frozen path properly. If not, the levers are
`SLICES_PER_SERIES`, `MAX_SERIES`, fewer folds, or freezing the backbone's
early stages.

**The focal findings.** Fine-tuning should help meniscus, ligaments, and
fracture most, since those are what frozen natural-image features handle worst.
If they do not move, the limit is elsewhere: the mined labels, or 224px
resolution against the size of the structures.